**Table of contents**<a id='toc0_'></a>    
- 1. [Toàn bộ Phi tài chính](#toc1_)    
- 2. [BDS](#toc2_)    
  - 2.1. [Toàn ngành](#toc2_1_)    
  - 2.2. [Doanh nghiệp](#toc2_2_)    
- 3. [CN](#toc3_)    
  - 3.1. [Toàn ngành](#toc3_1_)    
  - 3.2. [Ngành cấp 3](#toc3_2_)    
  - 3.3. [CB](#toc3_3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import sys
from dash import Dash, html, dcc, callback, Output, Input


In [3]:
# syspath = r"D:\OneDrive\Data\Python_custom_respository"
# if syspath not in sys.path:
#     sys.path.append(syspath)

# from utils.drawing import FPTSTemplate, FPTSBlue, FPTSGreen

# pio.templates["FPTS"] = FPTSTemplate
# pio.templates.default = "FPTS"


In [4]:
PATH = r"D:\00.data\datanormalized_24Q4.xlsx"
PL_SHEET_NAME = "ptcpl"
BS_SHEET_NAME = "ptcbs"
REVENUE = "Doanh thu thuần"
NI = "Lợi nhuận sau thuế thu nhập doanh nghiệp"
REV_GROWTH = "Tăng trưởng doanh thu thuần YoY"
NI_GROWTH = "Tăng trưởng LNST YoY"
GP = "Lợi nhuận gộp"
NIM = "Biên LNST"
GM = "Biên lợi nhuận gộp"
GICS1 = "GICS1_name"
GICS3 = "GICS3_name"
NUMBER_OF_PERIODS = 20
CURRENT_DATE = pd.to_datetime("2024-12-31")


In [5]:
def logit(x):
    return 1 / (10 ** (-x) + 1) - 0.5


In [6]:
logit_ticklabel = np.hstack([[-100, -2], np.linspace(-1, 1, 11), [2, 100]]).round(1)
logit_tickval = logit(logit_ticklabel)


In [7]:
def cagr(data: pd.DataFrame):
    n = data.shape[0]
    return (data.iloc[-1] / data.iloc[0]) ** (4 / n) - 1


In [8]:
def current_yoy_growth(data: pd.DataFrame):
    yoy_growth: pd.DataFrame = data.groupby(level=0).pct_change(4)
    current_growth: pd.DataFrame = yoy_growth.loc(axis=0)[:, CURRENT_DATE]
    current_growth.index = current_growth.index.get_level_values(0)
    return current_growth


def summary(data: pd.DataFrame):
    current_growth = current_yoy_growth(data)
    current_ni: pd.DataFrame = data.loc(axis=0)[:, CURRENT_DATE][NI]
    current_ni.index = current_ni.index.get_level_values(0)
    summary = current_growth.rename(columns={REVENUE: REV_GROWTH, NI: NI_GROWTH}).join(
        current_ni
    )
    return summary


# 1. <a id='toc1_'></a>[Toàn bộ Phi tài chính](#toc0_)

In [9]:
ptcpl = pd.read_excel(
    PATH, sheet_name=PL_SHEET_NAME, header=0, index_col=0, parse_dates=True
)
ptcpl


,Ticker,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,AAA,1573.567524,1.220192,1572.347332,1393.581780,178.765552,36.043002,56.993724,50.524147,0.19102,...,11.296470,-0.044342,62.658416,4.173949,58.484467,0.000000e+00,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Bao bì và đóng gói
2020-03-31,AAM,41.176955,0.000000,41.176955,36.136953,5.040002,0.647222,0.009707,0.008899,0.00000,...,0.089671,0.000000,0.665805,0.000000,0.665805,5.300000e-08,5.300000e-08,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
2020-03-31,AAT,45.982506,0.000000,45.982506,38.706874,7.275633,0.034282,3.036461,3.013552,0.00000,...,2.053040,0.000000,8.195431,0.000000,8.195431,2.360000e-07,2.360000e-07,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
2020-03-31,AAV,91.801697,0.000000,91.801697,81.492688,10.309009,0.564761,1.186606,1.174715,0.00000,...,1.429428,0.000000,4.366043,0.585432,3.780611,0.000000e+00,0.000000e+00,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,ABR,30.438412,0.000000,30.438412,12.763129,17.675283,1.990560,0.005197,0.000000,0.00000,...,3.073961,0.000000,10.666192,0.000000,10.666192,5.330000e-07,0.000000e+00,Dịch vụ viễn thông,Dịch vụ viễn thông,Dịch vụ viễn thông đa ngành
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,X20,575.541856,0.000000,575.541856,446.470021,129.071836,0.705559,0.000654,0.000000,0.00000,...,5.506866,-0.309897,29.766422,0.000000,29.766422,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
2024-12-31,XHC,152.766877,2.738176,150.028701,125.484051,24.544650,0.813653,3.883508,3.288618,0.00000,...,0.569962,0.000000,2.535927,0.000000,2.535927,1.200000e-07,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,Đồ gia dụng lâu bền
2024-12-31,XMC,325.131891,0.003594,325.128297,288.403841,36.724455,2.128890,6.545683,9.935908,0.00000,...,2.156000,0.000000,6.348060,1.370134,4.977926,0.000000e+00,0.000000e+00,Công nghiệp,Tư liệu sản xuất,Xây dựng


In [10]:
revni_ptc = ptcpl.groupby(ptcpl.index)[[REVENUE, NI]].sum()
revni_ptc


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,5.879148e+05,29418.730805
2020-06-30,5.867090e+05,36471.905765
2020-09-30,6.478115e+05,48637.360251
2020-12-31,7.214804e+05,68310.088641
2021-03-31,6.556824e+05,51952.326042
2021-06-30,7.770579e+05,72125.299754
2021-09-30,6.653345e+05,57870.637697
2021-12-31,8.793752e+05,78990.196646
2022-03-31,8.093343e+05,73710.326956


In [11]:
cagr_ptc = cagr(revni_ptc)
cagr_ptc


Doanh thu thuần                             0.124001
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.220997
dtype: float64

In [12]:
revni_ptc_yoy = revni_ptc.pct_change(4)
revni_ptc_yoy_current = revni_ptc_yoy.loc[CURRENT_DATE]

revni_ptc_yoy_current


Doanh thu thuần                             0.152082
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.499936
Name: 2024-12-31 00:00:00, dtype: float64

In [13]:
revni_sector = ptcpl.groupby([GICS1, ptcpl.index])[[REVENUE, NI]].sum()
revni_sector


Doanh thu thuần  \
GICS1_name   fiscalDate                    
Bất động sản 2020-03-31     40018.909668   
             2020-06-30     61370.289655   
             2020-09-30     91084.102028   
             2020-12-31     94153.042894   
             2021-03-31     66485.271667   
...                                  ...   
Y tế         2023-12-31     11690.188816   
             2024-03-31      9263.174605   
             2024-06-30      9995.454491   
             2024-09-30      9251.793946   
             2024-12-31     12064.146792   

                         Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS1_name   fiscalDate                                            
Bất động sản 2020-03-31                              11639.512040  
             2020-06-30                               8407.763939  
             2020-09-30                              15139.297543  
             2020-12-31                              21563.811871  
             2021-03-31                              12997.787439  
...                                                           ...  
Y tế         2023-12-31                               1008.183086  
             2024-03-31                                936.827837  
             2024-06-30                                976.299332  
             2024-09-30                                862.529658  
             2024-12-31                               1050.968860  

[220 rows x 2 columns]

In [14]:
revni_sector[revni_sector.index.get_level_values('GICS1_name') == 'Bất động sản']

Doanh thu thuần  \
GICS1_name   fiscalDate                    
Bất động sản 2020-03-31     40018.909668   
             2020-06-30     61370.289655   
             2020-09-30     91084.102028   
             2020-12-31     94153.042894   
             2021-03-31     66485.271667   
             2021-06-30     95483.036362   
             2021-09-30     72146.854399   
             2021-12-31     97358.616573   
             2022-03-31     52314.968277   
             2022-06-30     50190.637177   
             2022-09-30     78858.553231   
             2022-12-31    101936.831852   
             2023-03-31     86316.024877   
             2023-06-30    104601.499447   
             2023-09-30    102514.820386   
             2023-12-31     67403.360577   
             2024-03-31     49961.719567   
             2024-06-30     94425.465515   
             2024-09-30    120238.219692   
             2024-12-31    137753.810824   

                         Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS1_name   fiscalDate                                            
Bất động sản 2020-03-31                              11639.512040  
             2020-06-30                               8407.763939  
             2020-09-30                              15139.297543  
             2020-12-31                              21563.811871  
             2021-03-31                              12997.787439  
             2021-06-30                              18030.065833  
             2021-09-30                              15953.825585  
             2021-12-31                              10498.205084  
             2022-03-31                              11725.884619  
             2022-06-30                               8985.665023  
             2022-09-30                              22648.046949  
             2022-12-31                              10729.540129  
             2023-03-31                              16409.228090  
             2023-06-30                              14982.647254  
             2023-09-30                              15328.796136  
             2023-12-31                              10886.291742  
             2024-03-31                               5820.973267  
             2024-06-30                              17410.422441  
             2024-09-30                              18680.466018  
             2024-12-31                              22807.804480

In [15]:
current_summary = summary(revni_sector)
current_summary


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS1_name,,,
Bất động sản,1.043723,1.095094,22807.804480
Công nghiệp,0.134268,1.292740,14847.764039
Công nghệ,0.152931,0.378242,2462.467070
Dịch vụ công cộng,0.118754,0.091078,5123.444022
Dịch vụ viễn thông,0.213877,2.168756,4775.610464
Hàng tiêu dùng không thiết yếu,0.105608,0.743612,7304.475255
Hàng tiêu dùng thiết yếu,0.056379,0.165163,10222.097815
Nguyên vật liệu,0.081741,0.002192,6864.513528
Năng lượng,0.018715,-0.383629,4218.735855


In [16]:
current_summary["Tăng trưởng LNST YoY"].sort_values(ascending=False)

GICS1_name
Dịch vụ viễn thông                2.168756
Công nghiệp                       1.292740
Bất động sản                      1.095094
Hàng tiêu dùng không thiết yếu    0.743612
Công nghệ                         0.378242
Hàng tiêu dùng thiết yếu          0.165163
Dịch vụ công cộng                 0.091078
Y tế                              0.042438
Nguyên vật liệu                   0.002192
Tài chính                        -0.247642
Năng lượng                       -0.383629
Name: Tăng trưởng LNST YoY, dtype: float64

In [17]:
fig = px.scatter(
    current_summary,
    x=REV_GROWTH,
    y=NI_GROWTH,
    text=current_summary.index,
    size=NI,
    # template=FPTSTemplate,
)
textposi = ["middle right"] * 10
textposi[8:10] = ["middle left"] * 2
textposi[6] = "middle left"
textposi[0] = "middle left"
fig.update_traces(
    textfont={"size": 18},
    textposition=textposi,
    # marker={"color": [FPTSGreen] * 2 + [FPTSBlue[0]] * 8},
)
fig.update_layout(
    title="Tương quan tăng trưởng Doanh thu thuần và LNST Q4/2024",
    xaxis={"tickformat": ".0%"},
    yaxis={"tickformat": ".0%"},
)


In [18]:
px.bar(revni_ptc[REVENUE])


In [19]:
px.bar(revni_ptc[NI])


In [20]:
app = Dash()
app.layout = [
    html.Div(
        [
            html.H2(
                "Doanh thu và LNST các ngành cấp 1",
                style={"textAlign": "center", "color": "white"},
            ),
            dcc.Dropdown(
                revni_sector.index.unique(0), "Bất động sản", id="Ngành cấp 1"
            ),
            dcc.Graph(id="graph-content"),
        ]
    )
]


@callback(Output("graph-content", "figure"), Input("Ngành cấp 1", "value"))
def plot_revni(sector: str):
    data: pd.DataFrame = revni_sector.loc(axis=0)[sector]
    data.index = data.index.get_level_values(0)
    fig = make_subplots(rows=2, cols=1, subplot_titles=["Doanh thu", "LNST"])
    fig.add_trace(go.Bar(x=data.index, y=data[REVENUE]), row=1, col=1)
    fig.add_trace(go.Bar(x=data.index, y=data[NI]), row=2, col=1)
    fig.update_layout(margin=go.layout.Margin(l=50, r=50, b=50, t=50), showlegend=False)
    return fig


if __name__ == "__main__":
    app.run(debug=True)


# 2. <a id='toc2_'></a>[Bất động sản](#toc0_)

## 2.1. <a id='toc2_1_'></a>[Toàn ngành](#toc0_)

In [21]:
revni_bds = revni_sector.loc["Bất động sản"]
revni_bds


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,40018.909668,11639.512040
2020-06-30,61370.289655,8407.763939
2020-09-30,91084.102028,15139.297543
2020-12-31,94153.042894,21563.811871
2021-03-31,66485.271667,12997.787439
2021-06-30,95483.036362,18030.065833
2021-09-30,72146.854399,15953.825585
2021-12-31,97358.616573,10498.205084
2022-03-31,52314.968277,11725.884619


In [22]:
cagr_bds = cagr(revni_bds)
cagr_bds


Doanh thu thuần                             0.280465
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.144010
dtype: float64

In [23]:
px.bar(revni_bds[REVENUE])


In [24]:
px.bar(revni_bds[NI])


In [25]:
nim_bds = revni_bds[NI] / revni_bds[REVENUE]
nim_bds.name = NIM
nim_bds


fiscalDate
2020-03-31    0.290850
2020-06-30    0.137001
2020-09-30    0.166212
2020-12-31    0.229029
2021-03-31    0.195499
2021-06-30    0.188830
2021-09-30    0.221130
2021-12-31    0.107830
2022-03-31    0.224140
2022-06-30    0.179031
2022-09-30    0.287198
2022-12-31    0.105257
2023-03-31    0.190106
2023-06-30    0.143235
2023-09-30    0.149528
2023-12-31    0.161510
2024-03-31    0.116509
2024-06-30    0.184383
2024-09-30    0.155362
2024-12-31    0.165569
Name: Biên LNST, dtype: float64

In [26]:
px.line(nim_bds)


In [27]:
ptcbs = pd.read_excel(
    PATH, sheet_name=BS_SHEET_NAME, header=0, index_col=0, parse_dates=True
)
ptcbs


,Ticker,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,...,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,AAA,4556.885713,288.449897,196.124417,92.325479,840.336029,0.000000,0.00000,840.336029,2157.057260,...,0.0,0.0,0.0,0.0,0.0,0.0,7768.231727,Nguyên vật liệu,Vật liệu xây dựng,Bao bì và đóng gói
2020-03-31,AAM,177.906017,13.239992,6.239992,7.000000,32.534620,6.634960,-1.10034,27.000000,17.335429,...,NaN,0.0,NaN,0.0,0.0,0.0,219.150818,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
2020-03-31,AAT,150.202350,5.321284,5.321284,0.000000,1.000000,0.000000,0.00000,1.000000,128.711346,...,0.0,0.0,0.0,0.0,0.0,0.0,570.322269,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
2020-03-31,AAV,344.677555,2.596570,2.596570,0.000000,0.000000,0.000000,0.00000,0.000000,313.652243,...,0.0,0.0,0.0,0.0,0.0,0.0,596.584502,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,ABR,175.662863,67.624532,47.095230,20.529302,74.794367,0.000000,0.00000,74.794367,30.815600,...,0.0,0.0,0.0,0.0,0.0,0.0,262.948901,Dịch vụ viễn thông,Dịch vụ viễn thông,Dịch vụ viễn thông đa ngành
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,X20,502.200532,231.086175,231.086175,0.000000,0.400000,0.000000,0.00000,0.400000,78.512512,...,0.0,0.0,0.0,0.0,0.0,0.0,692.319833,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
2024-12-31,XHC,475.161555,32.823740,15.400550,17.423190,0.000000,0.000000,0.00000,0.000000,265.923300,...,0.0,0.0,0.0,0.0,0.0,0.0,677.367620,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,Đồ gia dụng lâu bền
2024-12-31,XMC,2240.210976,70.261646,65.261646,5.000000,10.041089,0.041089,0.00000,10.000000,963.965517,...,0.0,0.0,0.0,0.0,0.0,0.0,2945.898137,Công nghiệp,Tư liệu sản xuất,Xây dựng


In [28]:
bds_bs = ptcbs[ptcbs[GICS1] == "Bất động sản"]
bds_bs


,Ticker,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,...,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,AAV,344.677555,2.596570,2.596570,0.000000,0.000000,0.000000,0.000000,0.000000,313.652243,...,0.0,0.0,0.0,0.0,0.0,0.0,596.584502,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,AGG,4778.741697,271.036728,85.262582,185.774146,139.944104,0.000000,0.000000,139.944104,1094.368324,...,0.0,0.0,0.0,0.0,0.0,0.0,5630.751055,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,API,2352.665026,135.612729,45.042883,90.569845,186.453173,6.433165,-2.820376,182.840385,758.054715,...,0.0,0.0,0.0,0.0,0.0,0.0,2661.632905,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,BAX,339.986118,82.987530,4.887530,78.100000,248.200000,0.000000,0.000000,248.200000,7.518559,...,0.0,0.0,0.0,0.0,0.0,0.0,889.417813,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,BCM,28341.743841,320.329774,221.729774,98.600000,334.794890,0.000000,0.000000,334.794890,4080.088752,...,0.0,0.0,0.0,0.0,0.0,0.0,44024.502413,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,VPH,1335.992085,189.927194,33.064786,156.862408,0.131688,1.440800,-1.309112,0.000000,907.207870,...,0.0,0.0,0.0,0.0,0.0,0.0,1930.771960,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2024-12-31,VPI,5152.291995,457.372056,148.892993,308.479062,40.507890,0.000000,0.000000,40.507890,1680.117118,...,0.0,0.0,0.0,0.0,0.0,0.0,11144.158491,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2024-12-31,VRE,12312.424000,2884.680000,2884.680000,0.000000,125.918000,0.000000,0.000000,125.918000,2016.405000,...,0.0,0.0,0.0,0.0,0.0,0.0,55226.155000,Bất động sản,Quản lý và phát triển bất động sản,Vận hành bất động sản


In [29]:
VHM_bs = bds_bs[bds_bs["Ticker"] == "VHM"]
VHM_bs


,Ticker,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,...,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,VHM,141421.460,4439.856,993.105,3446.751,353.741,0.000,0.0,353.741,53058.225,...,0.0,0.0,0.0,0.0,0.0,0.0,203007.149,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-06-30,VHM,141342.852,11859.527,2458.360,9401.167,503.975,0.000,0.0,503.975,53621.486,...,0.0,0.0,0.0,0.0,0.0,0.0,224789.551,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-09-30,VHM,141229.902,13547.878,4779.171,8768.707,2449.214,1065.202,0.0,1384.012,46862.595,...,0.0,0.0,0.0,0.0,0.0,0.0,220509.139,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-12-31,VHM,102010.303,12479.392,4143.874,8335.518,3295.387,373.170,0.0,2922.217,34023.922,...,0.0,0.0,0.0,0.0,0.0,0.0,214937.493,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2021-03-31,VHM,94043.347,7746.445,3309.282,4437.163,1795.510,0.000,0.0,1795.510,34311.891,...,0.0,0.0,0.0,0.0,0.0,0.0,207714.674,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2021-06-30,VHM,101733.721,8417.973,2363.795,6054.178,5629.801,558.316,0.0,5071.485,42835.185,...,0.0,0.0,0.0,0.0,0.0,0.0,210849.493,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2021-09-30,VHM,102471.473,4504.657,2633.827,1870.830,7097.859,3605.821,0.0,3492.038,50375.165,...,0.0,0.0,0.0,0.0,0.0,0.0,219638.744,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2021-12-31,VHM,91216.562,4823.696,589.818,4233.878,4781.458,2147.535,0.0,2633.923,37928.893,...,0.0,0.0,0.0,0.0,0.0,0.0,230417.689,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2022-03-31,VHM,94888.743,5628.313,2553.086,3075.227,2621.355,0.000,0.0,2621.355,44040.609,...,0.0,0.0,0.0,0.0,0.0,0.0,233962.584,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp


In [30]:
bds_exVHM_bs = bds_bs[bds_bs["Ticker"] != "VHM"]
bds_exVHM_bs


,Ticker,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,...,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,AAV,344.677555,2.596570,2.596570,0.000000,0.000000,0.000000,0.000000,0.000000,313.652243,...,0.0,0.0,0.0,0.0,0.0,0.0,596.584502,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,AGG,4778.741697,271.036728,85.262582,185.774146,139.944104,0.000000,0.000000,139.944104,1094.368324,...,0.0,0.0,0.0,0.0,0.0,0.0,5630.751055,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,API,2352.665026,135.612729,45.042883,90.569845,186.453173,6.433165,-2.820376,182.840385,758.054715,...,0.0,0.0,0.0,0.0,0.0,0.0,2661.632905,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,BAX,339.986118,82.987530,4.887530,78.100000,248.200000,0.000000,0.000000,248.200000,7.518559,...,0.0,0.0,0.0,0.0,0.0,0.0,889.417813,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2020-03-31,BCM,28341.743841,320.329774,221.729774,98.600000,334.794890,0.000000,0.000000,334.794890,4080.088752,...,0.0,0.0,0.0,0.0,0.0,0.0,44024.502413,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,VPH,1335.992085,189.927194,33.064786,156.862408,0.131688,1.440800,-1.309112,0.000000,907.207870,...,0.0,0.0,0.0,0.0,0.0,0.0,1930.771960,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2024-12-31,VPI,5152.291995,457.372056,148.892993,308.479062,40.507890,0.000000,0.000000,40.507890,1680.117118,...,0.0,0.0,0.0,0.0,0.0,0.0,11144.158491,Bất động sản,Quản lý và phát triển bất động sản,Bất động sản phức hợp
2024-12-31,VRE,12312.424000,2884.680000,2884.680000,0.000000,125.918000,0.000000,0.000000,125.918000,2016.405000,...,0.0,0.0,0.0,0.0,0.0,0.0,55226.155000,Bất động sản,Quản lý và phát triển bất động sản,Vận hành bất động sản


In [31]:
bds_bs_sum = bds_bs.iloc[:, 1:-3].groupby(bds_bs.index).sum()
bds_bs_sum


,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,Phải thu khách hàng,...,Lợi nhuận sau thuế chưa phân phối,Lợi ích cổ đông không kiểm soát,Quỹ hỗ trợ sắp xếp doanh nghiệp,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,6.676345e+05,36394.138498,18348.172194,18045.966303,34141.097276,10578.980956,-372.305453,23934.421773,203595.618896,52199.916406,...,81680.815671,73322.293236,0.000000,10.111451,52.216172,0.000000,12.661770,39.554402,0.0,1.156302e+06
2020-06-30,6.753186e+05,63762.820480,24192.567504,39570.252977,27547.942811,3609.189790,-136.851371,24075.604392,208834.268609,52539.563717,...,87264.206515,70573.973298,0.000000,9.980798,51.718358,0.000000,12.911165,38.807193,0.0,1.209222e+06
2020-09-30,7.077836e+05,75384.674860,31723.485377,43661.189483,34300.190291,5696.832834,-108.358307,28711.715764,203606.701324,64720.215830,...,100070.431670,73055.828822,0.104636,9.850146,50.998836,0.000000,12.938849,38.059986,0.0,1.249184e+06
2020-12-31,6.683306e+05,77386.266729,30692.075027,46694.191702,42882.660519,10919.630490,-127.983520,32091.013549,192859.016461,54071.467233,...,112772.731600,88559.820986,0.000000,6.663281,46.386046,-2.003508,11.076775,37.312780,0.0,1.276509e+06
2021-03-31,6.799354e+05,67016.009940,31964.218757,35051.791183,37025.468537,7014.379520,-103.891219,30114.980236,206403.672676,60067.244343,...,124559.960034,95430.359235,0.000000,6.532628,47.642348,0.000000,11.076775,36.565574,0.0,1.294207e+06
2021-06-30,7.155565e+05,70673.186196,31108.754204,39564.431991,46346.430838,6916.592916,-124.095477,39553.933398,220123.458559,61223.869575,...,134628.457948,95338.066614,0.000000,6.401976,46.895142,0.000000,11.076775,35.818367,0.0,1.328741e+06
2021-09-30,7.596225e+05,71209.184518,36696.440309,34512.744209,45544.004527,13025.550731,-133.069671,32651.523466,260342.922975,63472.045926,...,132350.826612,108473.385301,0.000000,6.271323,46.147936,0.000000,11.076775,35.071161,0.0,1.381525e+06
2021-12-31,7.403901e+05,72768.374106,32039.612992,40728.761114,47007.146597,11896.892067,3035.871654,32074.382876,242653.024285,66307.584826,...,147101.194521,101424.493722,0.000000,6.140670,45.400729,0.000000,11.076775,34.323955,0.0,1.410417e+06
2022-03-31,7.756961e+05,71521.990165,26076.515248,45445.474917,40886.187008,7156.800600,-80.725577,33810.111985,266632.302716,67961.007969,...,154032.225369,78164.482058,0.000000,6.010018,44.653523,0.000000,11.076775,33.576748,0.0,1.465951e+06


In [32]:
bds_exVHM_bs_sum = bds_exVHM_bs.iloc[:, 1:-3].groupby(bds_exVHM_bs.index).sum()
bds_exVHM_bs_sum


,Tài sản ngắn hạn,Tiền và các khoản tương đương tiền,Tiền,Các khoản tương đương tiền,Các khoản đầu tư tài chính ngắn hạn,Đầu tư ngắn hạn,Dự phòng giảm giá đầu tư ngắn hạn,Đầu tư giữ đến ngày đáo hạn,Các khoản phải thu ngắn hạn,Phải thu khách hàng,...,Lợi nhuận sau thuế chưa phân phối,Lợi ích cổ đông không kiểm soát,Quỹ hỗ trợ sắp xếp doanh nghiệp,Nguồn vốn đầu tư XDCB,Nguồn kinh phí và quỹ khác,"Quỹ khen thưởng, phúc lợi (trước 2010)",Vốn ngân sách nhà nước,Nguồn kinh phí đã hình thành TSCĐ,Lợi ích của cổ đông không kiểm soát (trước 2015),TỔNG CỘNG NGUỒN VỐN
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,5.262131e+05,31954.282498,17355.067194,14599.215303,33787.356276,10578.980956,-372.305453,23580.680773,150537.393896,42235.620406,...,48797.306671,63906.761236,0.000000,10.111451,52.216172,0.000000,12.661770,39.554402,0.0,9.532952e+05
2020-06-30,5.339757e+05,51903.293480,21734.207504,30169.085977,27043.967811,3609.189790,-136.851371,23571.629392,155212.782609,42543.845717,...,49031.216515,63087.575298,0.000000,9.980798,51.718358,0.000000,12.911165,38.807193,0.0,9.844329e+05
2020-09-30,5.665537e+05,61836.796860,26944.314377,34892.482483,31850.976291,4631.630834,-108.358307,27327.703764,156744.106324,49599.045830,...,55234.783670,69180.148822,0.104636,9.850146,50.998836,0.000000,12.938849,38.059986,0.0,1.028675e+06
2020-12-31,5.663203e+05,64906.874729,26548.201027,38358.673702,39587.273519,10546.460490,-127.983520,29168.796549,158835.094461,43442.747233,...,56264.005600,85021.970986,0.000000,6.663281,46.386046,-2.003508,11.076775,37.312780,0.0,1.061571e+06
2021-03-31,5.858920e+05,59269.564940,28654.936757,30614.628183,35229.958537,7014.379520,-103.891219,28319.470236,172091.781676,49032.793343,...,62904.406034,91922.342235,0.000000,6.532628,47.642348,0.000000,11.076775,36.565574,0.0,1.086492e+06
2021-06-30,6.138227e+05,62255.213196,28744.959204,33510.253991,40716.629838,6358.276916,-124.095477,34482.448398,177288.273559,49635.087575,...,67571.192948,91712.568614,0.000000,6.401976,46.895142,0.000000,11.076775,35.818367,0.0,1.117891e+06
2021-09-30,6.571510e+05,66704.527518,34062.613309,32641.914209,38446.145527,9419.729731,-133.069671,29159.485466,209967.757975,52129.468926,...,63613.303612,103001.695301,0.000000,6.271323,46.147936,0.000000,11.076775,35.071161,0.0,1.161887e+06
2021-12-31,6.491736e+05,67944.678106,31449.794992,36494.883114,42225.688597,9749.357067,3035.871654,29440.459876,204724.131285,51093.692826,...,64051.095521,98108.483722,0.000000,6.140670,45.400729,0.000000,11.076775,34.323955,0.0,1.180000e+06
2022-03-31,6.808073e+05,65893.677165,23523.429248,42370.247917,38264.832008,7156.800600,-80.725577,31188.756985,222591.693716,50623.275969,...,70139.164369,71313.699058,0.000000,6.010018,44.653523,0.000000,11.076775,33.576748,0.0,1.231989e+06


In [33]:
CA = "Tài sản ngắn hạn"
CL = "Nợ ngắn hạn"


def current_ratio(data: pd.DataFrame):
    return data[CA] / data[CL]


bds_cr = current_ratio(bds_bs_sum)
bds_cr


fiscalDate
2020-03-31    1.373650
2020-06-30    1.337080
2020-09-30    1.452546
2020-12-31    1.434790
2021-03-31    1.493450
2021-06-30    1.558553
2021-09-30    1.755536
2021-12-31    1.725924
2022-03-31    1.770903
2022-06-30    1.446557
2022-09-30    1.399545
2022-12-31    1.358688
2023-03-31    1.362528
2023-06-30    1.306254
2023-09-30    1.324963
2023-12-31    1.299138
2024-03-31    1.313607
2024-06-30    1.286915
2024-09-30    1.296680
2024-12-31    1.174232
dtype: float64

In [34]:
VHM_cr = current_ratio(VHM_bs)
VHM_cr


fiscalDate
2020-03-31    1.167693
2020-06-30    1.133434
2020-09-30    1.192046
2020-12-31    1.006925
2021-03-31    1.008325
2021-06-30    1.071799
2021-09-30    1.260395
2021-12-31    1.214899
2022-03-31    1.342643
2022-06-30    1.048945
2022-09-30    1.002252
2022-12-31    0.989675
2023-03-31    1.076433
2023-06-30    1.027009
2023-09-30    1.095786
2023-12-31    1.122558
2024-03-31    1.187560
2024-06-30    1.139401
2024-09-30    1.227799
2024-12-31    0.990066
dtype: float64

In [35]:
bds_exVHM_cr = current_ratio(bds_exVHM_bs_sum)
bds_exVHM_cr


fiscalDate
2020-03-31    1.442005
2020-06-30    1.403845
2020-09-30    1.536233
2020-12-31    1.553711
2021-03-31    1.618436
2021-06-30    1.685414
2021-09-30    1.870093
2021-12-31    1.834340
2022-03-31    1.853294
2022-06-30    1.584310
2022-09-30    1.532200
2022-12-31    1.485102
2023-03-31    1.458323
2023-06-30    1.394605
2023-09-30    1.399858
2023-12-31    1.356949
2024-03-31    1.354258
2024-06-30    1.335670
2024-09-30    1.318649
2024-12-31    1.241190
dtype: float64

In [36]:
DEPOSIT = "Người mua trả tiền trước"
DEPOSIT_LT = "Người mua trả trước dài hạn"
UNEARN_REV = "Doanh thu chưa thực hiện ngắn hạn"
UNEARN_REV_LT = "Doanh thu chưa thực hiện dài hạn"
INV = "Hàng tồn kho"


def dep2inv(data: pd.DataFrame):
    return (data[DEPOSIT] + data[DEPOSIT_LT]) / data[INV]


bds_dep2inv = dep2inv(bds_bs_sum)
bds_dep2inv


fiscalDate
2020-03-31    0.398057
2020-06-30    0.393203
2020-09-30    0.364454
2020-12-31    0.318347
2021-03-31    0.300345
2021-06-30    0.298235
2021-09-30    0.251358
2021-12-31    0.215809
2022-03-31    0.199251
2022-06-30    0.376701
2022-09-30    0.425847
2022-12-31    0.412076
2023-03-31    0.396859
2023-06-30    0.364482
2023-09-30    0.300106
2023-12-31    0.318072
2024-03-31    0.299208
2024-06-30    0.335285
2024-09-30    0.367035
2024-12-31    0.467787
dtype: float64

In [37]:
VHM_dep2inv = dep2inv(VHM_bs)
VHM_dep2inv


fiscalDate
2020-03-31    0.668729
2020-06-30    0.646730
2020-09-30    0.701962
2020-12-31    0.631776
2021-03-31    0.608074
2021-06-30    0.641828
2021-09-30    0.494613
2021-12-31    0.312381
2022-03-31    0.195352
2022-06-30    0.857083
2022-09-30    1.106046
2022-12-31    0.947137
2023-03-31    0.932118
2023-06-30    0.884927
2023-09-30    0.691479
2023-12-31    1.041285
2024-03-31    0.741216
2024-06-30    0.813334
2024-09-30    0.852369
2024-12-31    0.951969
dtype: float64

In [38]:
bds_exVHM_dep2inv = dep2inv(bds_exVHM_bs_sum)
bds_exVHM_dep2inv


fiscalDate
2020-03-31    0.338238
2020-06-30    0.340072
2020-09-30    0.307475
2020-12-31    0.270860
2021-03-31    0.257883
2021-06-30    0.258921
2021-09-30    0.227216
2021-12-31    0.207041
2022-03-31    0.199601
2022-06-30    0.302875
2022-09-30    0.328073
2022-12-31    0.323153
2023-03-31    0.314292
2023-06-30    0.289759
2023-09-30    0.244877
2023-12-31    0.225476
2024-03-31    0.237522
2024-06-30    0.270735
2024-09-30    0.308064
2024-12-31    0.414568
dtype: float64

In [39]:
def dep_unearn2inv(data: pd.DataFrame):
    return (
        data[DEPOSIT] + data[DEPOSIT_LT] + data[UNEARN_REV] + data[UNEARN_REV_LT]
    ) / data[INV]


In [40]:
bds_inv2dep_unearn = dep_unearn2inv(bds_bs_sum)
bds_inv2dep_unearn


fiscalDate
2020-03-31    0.507177
2020-06-30    0.502237
2020-09-30    0.479369
2020-12-31    0.460857
2021-03-31    0.420757
2021-06-30    0.418763
2021-09-30    0.371557
2021-12-31    0.336785
2022-03-31    0.311756
2022-06-30    0.468043
2022-09-30    0.517091
2022-12-31    0.498960
2023-03-31    0.486086
2023-06-30    0.459183
2023-09-30    0.395269
2023-12-31    0.405778
2024-03-31    0.381065
2024-06-30    0.415755
2024-09-30    0.443754
2024-12-31    0.551055
dtype: float64

In [41]:
bds_exVHM_dep_unearn2inv = dep_unearn2inv(bds_exVHM_bs_sum)
bds_exVHM_dep_unearn2inv


fiscalDate
2020-03-31    0.464427
2020-06-30    0.464704
2020-09-30    0.434699
2020-12-31    0.421291
2021-03-31    0.388306
2021-06-30    0.386960
2021-09-30    0.353386
2021-12-31    0.333226
2022-03-31    0.316943
2022-06-30    0.404041
2022-09-30    0.428410
2022-12-31    0.420602
2023-03-31    0.413225
2023-06-30    0.394139
2023-09-30    0.349104
2023-12-31    0.321253
2024-03-31    0.327874
2024-06-30    0.359608
2024-09-30    0.391955
2024-12-31    0.504952
dtype: float64

In [42]:
VHM_dep_unearn2inv = dep_unearn2inv(VHM_bs)
VHM_dep_unearn2inv


fiscalDate
2020-03-31    0.700613
2020-06-30    0.681334
2020-09-30    0.743969
2020-12-31    0.722000
2021-03-31    0.655939
2021-06-30    0.696710
2021-09-30    0.554639
2021-12-31    0.375993
2022-03-31    0.253936
2022-06-30    0.884494
2022-09-30    1.134033
2022-12-31    0.970451
2023-03-31    0.958425
2023-06-30    0.912213
2023-09-30    0.722407
2023-12-31    1.065960
2024-03-31    0.762205
2024-06-30    0.831573
2024-09-30    0.870066
2024-12-31    0.970497
dtype: float64

In [43]:
# liquidity_ratio = pd.concat([bds_cr, bds_inv2dep, bds_inv2dep_unearn], axis=1)
# liquidity_ratio.columns = [
#     "Tỷ số thanh toán hiện hành",
#     "Hàng tồn kho/Người mua trả trước",
#     "Hàng tồn kho/Người mua trả trước+DT chưa thực hiện",
# ]
# liquidity_ratio


## 2.2. <a id='toc2_2_'></a>[Doanh nghiệp](#toc0_)

In [44]:
revni_bds_ticker = (
    ptcpl.loc[ptcpl[GICS1] == "Bất động sản"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)[[REVENUE, NI]]
revni_bds_ticker


,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
AAV,2020-03-31,91.801697,4.366043
AGG,2020-03-31,42.853729,0.694985
API,2020-03-31,128.473885,11.751050
BAX,2020-03-31,16.719515,4.897992
BCM,2020-03-31,1229.362211,332.226605
...,...,...,...
VPH,2024-12-31,12.278783,-10.916157
VPI,2024-12-31,749.879238,101.198846
VRE,2024-12-31,2128.159000,1085.329000


In [45]:
revni_VHM = revni_bds_ticker.loc(axis=0)["VHM"]
revni_VHM


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,6519.224,7645.065
2020-06-30,16376.482,3416.257
2020-09-30,26482.588,6146.257
2020-12-31,21512.001,11559.552
2021-03-31,12986.441,5477.762
2021-06-30,28015.120,10602.296
2021-09-30,20679.434,11195.103
2021-12-31,23412.970,11986.291
2022-03-31,8923.490,4724.937


In [46]:
revni_bds_exVHM = revni_bds - revni_VHM
revni_bds_exVHM


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,33499.685668,3994.447040
2020-06-30,44993.807655,4991.506939
2020-09-30,64601.514028,8993.040543
2020-12-31,72641.041894,10004.259871
2021-03-31,53498.830667,7520.025439
2021-06-30,67467.916362,7427.769833
2021-09-30,51467.420399,4758.722585
2021-12-31,73945.646573,-1488.085916
2022-03-31,43391.478277,7000.947619


In [47]:
current_summary_bds = summary(revni_bds_ticker)
current_summary_bds


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,,
AAV,7.810838,0.347333,-6.485327
AGG,-0.075701,-0.806876,21.109491
API,0.562848,-1.590517,11.484367
BAX,-0.165384,-0.326442,6.229490
BCM,-0.604736,-0.248512,1540.356607
...,...,...,...
VPH,-0.544058,-1.476556,-10.916157
VPI,4.572849,3.052194,101.198846
VRE,-0.091568,0.016693,1085.329000


In [48]:
current_summary_bds_logit = logit(current_summary_bds)
current_summary_bds_logit


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,,
AAV,0.500000,0.189924,-0.500000
AGG,-0.043467,-0.365052,0.500000
API,0.285162,-0.474969,0.500000
BAX,-0.094069,-0.179540,0.499999
BCM,-0.300984,-0.139275,0.500000
...,...,...,...
VPH,-0.277774,-0.467701,-0.500000
VPI,0.499973,0.499114,0.500000
VRE,-0.052516,0.009608,0.500000


In [49]:
px.scatter(
    current_summary_bds_logit,
    x=REV_GROWTH,
    y=NI_GROWTH,
    size=current_summary_bds[NI].where(current_summary_bds[NI] > 1, 1),
    color=current_summary_bds_logit.index,
).update_layout(
    xaxis={"tickvals": logit_tickval, "ticktext": logit_ticklabel},
    yaxis={"tickvals": logit_tickval, "ticktext": logit_ticklabel},
)


# 3. <a id='toc3_'></a>[Công nghiệp](#toc0_)

## 3.1. <a id='toc3_1_'></a>[Toàn ngành](#toc0_)

In [50]:
revni_cn = revni_sector.loc["Công nghiệp"]
revni_cn


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,83248.338581,1190.015211
2020-06-30,74793.035166,1002.814051
2020-09-30,79192.937988,1332.826597
2020-12-31,100928.170419,7991.188103
2021-03-31,77144.329016,1833.867381
2021-06-30,95760.455059,4078.392631
2021-09-30,83791.889332,1010.040435
2021-12-31,109228.466221,9928.267577
2022-03-31,91636.372269,6701.680031


In [51]:
nim_cn = revni_cn[NI] / revni_cn[REVENUE]
nim_cn


fiscalDate
2020-03-31    0.014295
2020-06-30    0.013408
2020-09-30    0.016830
2020-12-31    0.079177
2021-03-31    0.023772
2021-06-30    0.042590
2021-09-30    0.012054
2021-12-31    0.090895
2022-03-31    0.073133
2022-06-30    0.081323
2022-09-30    0.059582
2022-12-31    0.015822
2023-03-31    0.058505
2023-06-30    0.079642
2023-09-30    0.053308
2023-12-31    0.042408
2024-03-31    0.112755
2024-06-30    0.100149
2024-09-30    0.081836
2024-12-31    0.085721
dtype: float64

In [52]:
cagr(revni_cn)


Doanh thu thuần                             0.157816
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.656615
dtype: float64

In [53]:
gp_cn = ptcpl.loc[ptcpl[GICS1] == "Công nghiệp", "Lợi nhuận gộp"].groupby(level=0).sum()
gp_cn


fiscalDate
2020-03-31     8054.683218
2020-06-30     3355.566658
2020-09-30     5313.863597
2020-12-31    12095.822639
2021-03-31     4321.780095
2021-06-30     7358.921420
2021-09-30     5988.011229
2021-12-31    11916.174647
2022-03-31    10824.012634
2022-06-30    17070.623256
2022-09-30    15544.238005
2022-12-31     9009.636057
2023-03-31    15824.705022
2023-06-30    16886.530492
2023-09-30    17170.866752
2023-12-31    15284.354671
2024-03-31    22007.302497
2024-06-30    22074.626866
2024-09-30    23214.993155
2024-12-31    25655.747129
Name: Lợi nhuận gộp, dtype: float64

In [54]:
gm_cn = gp_cn / revni_cn[REVENUE]
gm_cn


fiscalDate
2020-03-31    0.096755
2020-06-30    0.044865
2020-09-30    0.067100
2020-12-31    0.119846
2021-03-31    0.056022
2021-06-30    0.076847
2021-09-30    0.071463
2021-12-31    0.109094
2022-03-31    0.118119
2022-06-30    0.142546
2022-09-30    0.130475
2022-12-31    0.069386
2023-03-31    0.153034
2023-06-30    0.139325
2023-09-30    0.141057
2023-12-31    0.100090
2024-03-31    0.168574
2024-06-30    0.149776
2024-09-30    0.159824
2024-12-31    0.148119
dtype: float64

In [55]:
profitability_cn = pd.concat([gm_cn, nim_cn], axis=1)
profitability_cn.columns = ["Biên lợi nhuận gộp", "Biên LNST"]
profitability_cn


,Biên lợi nhuận gộp,Biên LNST
fiscalDate,,
2020-03-31,0.096755,0.014295
2020-06-30,0.044865,0.013408
2020-09-30,0.067100,0.016830
2020-12-31,0.119846,0.079177
2021-03-31,0.056022,0.023772
2021-06-30,0.076847,0.042590
2021-09-30,0.071463,0.012054
2021-12-31,0.109094,0.090895
2022-03-31,0.118119,0.073133


## 3.2. <a id='toc3_2_'></a>[Ngành cấp 3](#toc0_)

In [56]:
cn_pl = ptcpl[ptcpl[GICS1] == "Công nghiệp"]
cn_pl


,Ticker,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
fiscalDate,,,,,,,,,,,,,,,,,,,,,
2020-03-31,ACC,91.216268,0.024591,91.191677,69.438527,21.753150,0.068324,2.011843,2.011843,0.000000,...,1.981562,0.000000,7.936587,0.302348,7.634238,7.630000e-07,0.000000,Công nghiệp,Tư liệu sản xuất,Xây dựng
2020-03-31,ACV,3634.688857,0.169808,3634.519049,2011.662939,1622.856110,544.940065,41.571106,23.333959,44.204588,...,376.517399,0.000000,1550.206985,1.548267,1548.658717,7.110000e-07,0.000000,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ sân bay
2020-03-31,AMS,505.611622,0.000000,505.611622,473.548706,32.062916,2.811073,19.552455,16.106522,0.000000,...,1.130225,0.000000,4.520898,0.004563,4.516335,0.000000e+00,0.000000,Công nghiệp,Tư liệu sản xuất,Máy công nghiệp
2020-03-31,APH,1836.151100,1.396116,1834.754985,1611.236133,223.518851,37.812631,70.128798,63.656557,-4.816267,...,16.009566,-0.044342,41.842791,37.764131,4.078660,0.000000e+00,0.000000,Công nghiệp,Tư liệu sản xuất,Công nghiệp đa ngành
2020-03-31,ARM,60.697116,0.000000,60.697116,52.747961,7.949155,0.078689,1.103237,0.716591,0.000000,...,0.126729,0.000000,1.158017,0.000000,1.158017,4.470000e-07,0.000000,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ sân bay
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,VTO,294.072627,0.000000,294.072627,230.688702,63.383925,8.676392,12.041768,3.627124,0.000000,...,9.132822,0.012986,32.897728,0.000000,32.897728,1.490000e-07,0.000000,Công nghiệp,Vận tải,Vận tải biển
2024-12-31,VTP,5707.195285,0.000000,5707.195285,5363.231146,343.964139,19.254004,16.632297,16.587033,0.000000,...,36.359716,0.000000,130.389875,0.000000,130.389875,7.930000e-07,0.000000,Công nghiệp,Vận tải,Hỗ trợ vận tải
2024-12-31,VTX,65.267661,0.000000,65.267661,76.981029,-11.713368,1.359358,1.432424,1.432424,0.000000,...,0.000000,0.000000,-44.153383,0.000000,-44.153383,-2.105000e-06,-0.000002,Công nghiệp,Vận tải,Hỗ trợ vận tải


In [57]:
revni_cn_sector = cn_pl.groupby([GICS3, cn_pl.index])[[REVENUE, NI]].sum()
revni_cn_sector


Doanh thu thuần  \
GICS3_name           fiscalDate                    
Công nghiệp đa ngành 2020-03-31      2158.773191   
                     2020-06-30      2184.367400   
                     2020-09-30      2417.366674   
                     2020-12-31      2688.489692   
                     2021-03-31      2913.650695   
...                                          ...   
Xây dựng             2023-12-31     55010.692550   
                     2024-03-31     34834.268586   
                     2024-06-30     47371.214598   
                     2024-09-30     42535.981662   
                     2024-12-31     60262.935041   

                                 Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name           fiscalDate                                            
Công nghiệp đa ngành 2020-03-31                                104.057601  
                     2020-06-30                                109.397871  
                     2020-09-30                                141.031662  
                     2020-12-31                                186.006093  
                     2021-03-31                                138.435559  
...                                                                   ...  
Xây dựng             2023-12-31                               1456.105920  
                     2024-03-31                               1426.882723  
                     2024-06-30                               2248.738799  
                     2024-09-30                               1786.973571  
                     2024-12-31                               2587.305584  

[280 rows x 2 columns]

In [58]:
gp_cn_sector = cn_pl.groupby([GICS3, cn_pl.index])["Lợi nhuận gộp"].sum()
gp_cn_sector


GICS3_name            fiscalDate
Công nghiệp đa ngành  2020-03-31     247.461523
                      2020-06-30     216.971274
                      2020-09-30     281.316609
                      2020-12-31     311.548226
                      2021-03-31     348.220322
                                       ...     
Xây dựng              2023-12-31    4696.636374
                      2024-03-31    4334.706769
                      2024-06-30    4807.955273
                      2024-09-30    4960.572528
                      2024-12-31    5891.448423
Name: Lợi nhuận gộp, Length: 280, dtype: float64

In [59]:
gm_cn_sector = gp_cn_sector / revni_cn_sector[REVENUE]
gm_cn_sector


GICS3_name            fiscalDate
Công nghiệp đa ngành  2020-03-31    0.114631
                      2020-06-30    0.099329
                      2020-09-30    0.116373
                      2020-12-31    0.115882
                      2021-03-31    0.119513
                                      ...   
Xây dựng              2023-12-31    0.085377
                      2024-03-31    0.124438
                      2024-06-30    0.101495
                      2024-09-30    0.116621
                      2024-12-31    0.097762
Length: 280, dtype: float64

In [60]:
current_gm_cn_sector = gm_cn_sector.loc(axis=0)[:, CURRENT_DATE]
current_gm_cn_sector


GICS3_name                                 fiscalDate
Công nghiệp đa ngành                       2024-12-31    0.140356
Dịch vụ cảng biển                          2024-12-31    0.264171
Dịch vụ hỗ trợ khác                        2024-12-31    0.136781
Dịch vụ hỗ trợ việc làm và nguồn nhân lực  2024-12-31    0.333730
Dịch vụ môi trường                         2024-12-31    0.335423
Dịch vụ sân bay                            2024-12-31    0.526395
Dịch vụ tư vấn & nghiên cứu                2024-12-31    0.212885
Hỗ trợ vận tải                             2024-12-31    0.080739
Máy công nghiệp                            2024-12-31    0.111577
Thiết bị điện                              2024-12-31    0.193091
Vận tải biển                               2024-12-31    0.098991
Vận tải hàng không                         2024-12-31    0.116519
Vận tải đường bộ & đường sắt               2024-12-31    0.440965
Xây dựng                                   2024-12-31    0.097762
dtype: float64

In [61]:
nim_cn_sector = revni_cn_sector[NI] / revni_cn_sector[REVENUE]
nim_cn_sector


GICS3_name            fiscalDate
Công nghiệp đa ngành  2020-03-31    0.048202
                      2020-06-30    0.050082
                      2020-09-30    0.058341
                      2020-12-31    0.069186
                      2021-03-31    0.047513
                                      ...   
Xây dựng              2023-12-31    0.026470
                      2024-03-31    0.040962
                      2024-06-30    0.047471
                      2024-09-30    0.042011
                      2024-12-31    0.042934
Length: 280, dtype: float64

In [62]:
nim_cn_sector.loc["Dịch vụ cảng biển"]


fiscalDate
2020-03-31    0.088435
2020-06-30    0.145942
2020-09-30    0.088443
2020-12-31    0.069531
2021-03-31    0.171686
2021-06-30    0.225935
2021-09-30    0.221443
2021-12-31    0.306987
2022-03-31    0.233960
2022-06-30    0.261021
2022-09-30    0.210435
2022-12-31    0.128257
2023-03-31    0.182514
2023-06-30    0.373490
2023-09-30    0.177094
2023-12-31    0.137906
2024-03-31    0.206350
2024-06-30    0.226898
2024-09-30    0.208541
2024-12-31    0.169921
dtype: float64

In [63]:
growth_rate_cn_sector = revni_cn_sector.groupby(level=0).pct_change(4)
growth_rate_cn_sector


Doanh thu thuần  \
GICS3_name           fiscalDate                    
Công nghiệp đa ngành 2020-03-31              NaN   
                     2020-06-30              NaN   
                     2020-09-30              NaN   
                     2020-12-31              NaN   
                     2021-03-31         0.349679   
...                                          ...   
Xây dựng             2023-12-31         0.208340   
                     2024-03-31         0.585413   
                     2024-06-30         0.324949   
                     2024-09-30         0.237690   
                     2024-12-31         0.095477   

                                 Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name           fiscalDate                                            
Công nghiệp đa ngành 2020-03-31                                       NaN  
                     2020-06-30                                       NaN  
                     2020-09-30                                       NaN  
                     2020-12-31                                       NaN  
                     2021-03-31                                  0.330374  
...                                                                   ...  
Xây dựng             2023-12-31                                 -4.384263  
                     2024-03-31                                 24.818061  
                     2024-06-30                                  2.197491  
                     2024-09-30                                  1.597828  
                     2024-12-31                                  0.776866  

[280 rows x 2 columns]

In [64]:
current_growth_rate_cn_sector = growth_rate_cn_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_cn_sector.index = (
    current_growth_rate_cn_sector.index.get_level_values(GICS3)
)
current_growth_rate_cn_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Công nghiệp đa ngành,0.132997,-0.490837
Dịch vụ cảng biển,0.234187,0.520701
Dịch vụ hỗ trợ khác,0.071459,2.503360
Dịch vụ hỗ trợ việc làm và nguồn nhân lực,0.949799,0.652492
Dịch vụ môi trường,-0.297290,-0.122316
Dịch vụ sân bay,0.156201,0.924992
Dịch vụ tư vấn & nghiên cứu,0.069501,0.250351
Hỗ trợ vận tải,0.348024,-0.213235
Máy công nghiệp,0.108965,0.592489


In [65]:
current_ni_cn_sector = revni_cn_sector.loc(axis=0)[:, CURRENT_DATE][NI]
current_ni_cn_sector.index = current_ni_cn_sector.index.get_level_values(GICS3)
current_ni_cn_sector


GICS3_name
Công nghiệp đa ngành                          130.456880
Dịch vụ cảng biển                            2057.774316
Dịch vụ hỗ trợ khác                           435.145875
Dịch vụ hỗ trợ việc làm và nguồn nhân lực      13.111485
Dịch vụ môi trường                             33.947618
Dịch vụ sân bay                              3628.157617
Dịch vụ tư vấn & nghiên cứu                   157.056079
Hỗ trợ vận tải                                302.385324
Máy công nghiệp                              2615.400403
Thiết bị điện                                1214.955738
Vận tải biển                                  406.585550
Vận tải hàng không                           1031.517247
Vận tải đường bộ & đường sắt                  233.964323
Xây dựng                                     2587.305584
Name: Lợi nhuận sau thuế thu nhập doanh nghiệp, dtype: float64

In [66]:
current_cn_summary: pd.DataFrame = current_growth_rate_cn_sector.rename(
    columns={REVENUE: REV_GROWTH, NI: NI_GROWTH}
).join(current_ni_cn_sector)
current_cn_summary


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,,
Công nghiệp đa ngành,0.132997,-0.490837,130.456880
Dịch vụ cảng biển,0.234187,0.520701,2057.774316
Dịch vụ hỗ trợ khác,0.071459,2.503360,435.145875
Dịch vụ hỗ trợ việc làm và nguồn nhân lực,0.949799,0.652492,13.111485
Dịch vụ môi trường,-0.297290,-0.122316,33.947618
Dịch vụ sân bay,0.156201,0.924992,3628.157617
Dịch vụ tư vấn & nghiên cứu,0.069501,0.250351,157.056079
Hỗ trợ vận tải,0.348024,-0.213235,302.385324
Máy công nghiệp,0.108965,0.592489,2615.400403


In [67]:
fig2 = px.scatter(
    current_cn_summary,
    x=REV_GROWTH,
    y=NI_GROWTH,
    size=NI,
    color=current_cn_summary.index,
)
fig2.update_layout(title="Tương quan tăng trưởng Doanh thu thuần và LNST Q4/2024")


## 3.3. <a id='toc3_3_'></a>[Cảng biển](#toc0_)

In [68]:
cb_pl = (
    cn_pl[cn_pl[GICS3] == "Dịch vụ cảng biển"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
cb_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
CAG,2020-03-31,13.990874,0.000000,13.990874,11.219952,2.770922,0.570962,0.000000,0.000000,0.000000,0.357148,...,0.182329,0.000000,0.624315,0.000000,0.624315,4.500000e-08,4.500000e-08,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
CCT,2020-03-31,27.592568,0.000000,27.592568,21.551320,6.041248,0.150128,0.843877,0.843877,0.000000,0.204632,...,0.000000,0.000000,0.305845,0.000000,0.305845,0.000000e+00,0.000000e+00,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
CLL,2020-03-31,84.223373,0.000000,84.223373,55.571787,28.651585,2.617035,0.305332,0.305332,0.000000,0.000000,...,5.340445,0.000000,21.223782,0.545496,20.678285,6.080000e-07,0.000000e+00,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
CPI,2020-03-31,12.400039,0.000000,12.400039,11.481881,0.918158,0.064018,0.019208,0.016994,0.000000,0.000000,...,0.044291,0.000000,-0.266473,0.000000,-0.266473,-7.000000e-09,0.000000e+00,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
CQN,2020-03-31,198.789509,0.000000,198.789509,185.756405,13.033104,2.877105,0.006037,0.000000,0.000000,0.000000,...,0.596824,0.000000,2.387295,0.000000,2.387295,4.800000e-08,0.000000e+00,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
QNP,2024-12-31,258.787815,3.782991,255.004824,190.546050,64.458774,5.453513,4.637015,4.637015,0.000000,4.753885,...,7.286708,0.000000,27.495223,0.000000,27.495223,6.800000e-07,0.000000e+00,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
SGP,2024-12-31,291.696542,0.000000,291.696542,198.357020,93.339522,219.467288,207.743066,0.214187,31.014634,0.000000,...,34.142096,-16.986107,44.624857,-0.894253,45.519109,2.100000e-07,2.100000e-07,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển
TCL,2024-12-31,464.075725,0.000000,464.075725,383.971514,80.104211,2.729094,0.118144,0.118144,10.087878,8.418844,...,10.714294,0.000000,53.289252,0.057488,53.231764,1.556000e-06,1.556000e-06,Công nghiệp,Cơ sở hạ tầng giao thông vận tải,Dịch vụ cảng biển


In [69]:
revni_cb = cb_pl[[REVENUE, NI]]
cb_current_growth = current_yoy_growth(revni_cb)
cb_current_growth


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
CAG,0.031855,5.759313
CCT,0.146489,-0.725897
CLL,0.165585,-0.031604
CPI,0.122622,0.997276
CQN,0.172070,-0.006186
DS3,-0.012441,-0.395110
DVP,0.206872,0.124204
DXP,-0.436158,-0.284899
GMD,0.365583,0.856264


In [70]:
gm_cb = gm_cn_sector.loc["Dịch vụ cảng biển"]
gm_cb


fiscalDate
2020-03-31    0.231739
2020-06-30    0.241134
2020-09-30    0.223054
2020-12-31    0.238217
2021-03-31    0.261295
2021-06-30    0.321076
2021-09-30    0.313336
2021-12-31    0.301184
2022-03-31    0.315852
2022-06-30    0.341632
2022-09-30    0.319308
2022-12-31    0.271933
2023-03-31    0.287392
2023-06-30    0.283403
2023-09-30    0.254135
2023-12-31    0.218753
2024-03-31    0.266340
2024-06-30    0.243877
2024-09-30    0.263164
2024-12-31    0.264171
dtype: float64

# 4. <a id='toc4_'></a>[Hàng tiêu dùng không thiết yếu](#toc0_)

## 4.1. <a id='toc4_1_'></a>[Toàn ngành](#toc0_)

In [71]:
revni_htdkty = revni_sector.loc["Hàng tiêu dùng không thiết yếu"]
revni_htdkty

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,92163.249693,3394.198898
2020-06-30,83387.508097,2994.061340
2020-09-30,98135.644245,4184.253044
2020-12-31,109936.344006,7064.808062
2021-03-31,101593.469324,4909.360784
2021-06-30,108793.551853,4625.028536
2021-09-30,86629.455953,3560.735991
2021-12-31,136318.758211,9004.857400
2022-03-31,127305.040508,6379.568991


In [72]:
cagr(revni_htdkty)

Doanh thu thuần                             0.097139
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.165656
dtype: float64

In [73]:
nim_htdkty = revni_htdkty[NI] / revni_htdkty[REVENUE]
nim_htdkty

fiscalDate
2020-03-31    0.036828
2020-06-30    0.035905
2020-09-30    0.042637
2020-12-31    0.064263
2021-03-31    0.048324
2021-06-30    0.042512
2021-09-30    0.041103
2021-12-31    0.066057
2022-03-31    0.050112
2022-06-30    0.045054
2022-09-30    0.037058
2022-12-31    0.036644
2023-03-31    0.027561
2023-06-30    0.020662
2023-09-30    0.021756
2023-12-31    0.031614
2024-03-31    0.033650
2024-06-30    0.038057
2024-09-30    0.037002
2024-12-31    0.049857
dtype: float64

In [74]:
gp_htdkty = ptcpl.loc[ptcpl[GICS1] == "Hàng tiêu dùng không thiết yếu", "Lợi nhuận gộp"].groupby(level=0).sum()
gp_htdkty

fiscalDate
2020-03-31    13326.591931
2020-06-30    11962.301276
2020-09-30    14285.691664
2020-12-31    17549.779337
2021-03-31    16081.802186
2021-06-30    16365.724301
2021-09-30    14060.051946
2021-12-31    21164.168514
2022-03-31    19283.539163
2022-06-30    19583.359367
2022-09-30    19211.470243
2022-12-31    19877.314676
2023-03-31    14845.693866
2023-06-30    14562.974414
2023-09-30    16425.139408
2023-12-31    18348.036317
2024-03-31    17787.820543
2024-06-30    19706.906719
2024-09-30    19994.273615
2024-12-31    22780.614215
Name: Lợi nhuận gộp, dtype: float64

In [75]:
gm_htdkty = gp_htdkty / revni_htdkty[REVENUE]
gm_htdkty

fiscalDate
2020-03-31    0.144598
2020-06-30    0.143454
2020-09-30    0.145571
2020-12-31    0.159636
2021-03-31    0.158296
2021-06-30    0.150429
2021-09-30    0.162301
2021-12-31    0.155255
2022-03-31    0.151475
2022-06-30    0.154490
2022-09-30    0.150825
2022-12-31    0.149197
2023-03-31    0.134042
2023-06-30    0.127113
2023-09-30    0.135073
2023-12-31    0.138460
2024-03-31    0.150195
2024-06-30    0.149072
2024-09-30    0.146846
2024-12-31    0.155489
dtype: float64

## 4.2. <a id='toc4_2_'></a>[Ngành cấp 3](#toc0_)

In [76]:
htdkty_pl = ptcpl[ptcpl[GICS1] == "Hàng tiêu dùng không thiết yếu"]
revni_htdkty_sector = htdkty_pl.groupby([GICS3, htdkty_pl.index])[[REVENUE, NI]].sum()
revni_htdkty_sector

Doanh thu thuần  \
GICS3_name          fiscalDate                    
Bán lẻ              2020-03-31     64116.897726   
                    2020-06-30     55836.184339   
                    2020-09-30     61789.616802   
                    2020-12-31     69478.472002   
                    2021-03-31     71370.905716   
...                                         ...   
Đồ gia dụng lâu bền 2023-12-31      6967.129475   
                    2024-03-31      5365.186530   
                    2024-06-30      5754.376713   
                    2024-09-30      4645.322262   
                    2024-12-31      6797.750616   

                                Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name          fiscalDate                                            
Bán lẻ              2020-03-31                               1763.679459  
                    2020-06-30                               1226.083899  
                    2020-09-30                               1501.186102  
                    2020-12-31                               1784.017128  
                    2021-03-31                               2467.351519  
...                                                                  ...  
Đồ gia dụng lâu bền 2023-12-31                                203.447402  
                    2024-03-31                                202.119738  
                    2024-06-30                                205.223104  
                    2024-09-30                                141.807705  
                    2024-12-31                                125.542059  

[140 rows x 2 columns]

In [77]:
current_ni_htdkty_sector = revni_htdkty_sector.loc(axis=0)[:, CURRENT_DATE][NI]
current_ni_htdkty_sector.index = current_ni_htdkty_sector.index.get_level_values(GICS3)
current_ni_htdkty_sector

GICS3_name
Bán lẻ                                2147.671238
Dệt may & Trang phục, phụ kiện        1107.777883
Khách sạn, nhà hàng & Giải trí         311.480353
Lốp & Cao su                          3384.518137
Nhà sản xuất ô tô và thiết bị ô tô     -10.206478
Truyền thông và xuất bản               237.692061
Đồ gia dụng lâu bền                    125.542059
Name: Lợi nhuận sau thuế thu nhập doanh nghiệp, dtype: float64

In [78]:
#revni_htdkty_sector.xs("Khách sạn, nhà hàng & Giải trí", level=GICS3)

In [79]:
# htdkty_pl[htdkty_pl.index.isin(["2023-12-31", "2024-12-31"])][["Ticker",NI]].sort_values(by=NI, ascending=False)

In [80]:
growth_rate_htdkty_sector = revni_htdkty_sector.groupby(level=0).pct_change(4)
growth_rate_htdkty_sector

Doanh thu thuần  \
GICS3_name          fiscalDate                    
Bán lẻ              2020-03-31              NaN   
                    2020-06-30              NaN   
                    2020-09-30              NaN   
                    2020-12-31              NaN   
                    2021-03-31         0.113137   
...                                         ...   
Đồ gia dụng lâu bền 2023-12-31         0.136512   
                    2024-03-31         0.044152   
                    2024-06-30         0.194105   
                    2024-09-30         0.107188   
                    2024-12-31        -0.024311   

                                Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name          fiscalDate                                            
Bán lẻ              2020-03-31                                       NaN  
                    2020-06-30                                       NaN  
                    2020-09-30                                       NaN  
                    2020-12-31                                       NaN  
                    2021-03-31                                  0.398980  
...                                                                  ...  
Đồ gia dụng lâu bền 2023-12-31                                 -0.320349  
                    2024-03-31                                  0.025151  
                    2024-06-30                                  0.386841  
                    2024-09-30                                  0.616124  
                    2024-12-31                                 -0.382926  

[140 rows x 2 columns]

In [81]:
current_growth_rate_htdkty_sector = growth_rate_htdkty_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_htdkty_sector.index = (
    current_growth_rate_htdkty_sector.index.get_level_values(GICS3)
)
current_growth_rate_htdkty_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Bán lẻ,0.093021,1.170977
"Dệt may & Trang phục, phụ kiện",0.142237,1.079325
"Khách sạn, nhà hàng & Giải trí",0.221651,12.356954
Lốp & Cao su,0.145116,0.614673
Nhà sản xuất ô tô và thiết bị ô tô,0.216220,-1.057641
Truyền thông và xuất bản,0.267899,0.420614
Đồ gia dụng lâu bền,-0.024311,-0.382926


In [82]:
gp_htdkty_sector = htdkty_pl.groupby([GICS3, htdkty_pl.index])["Lợi nhuận gộp"].sum()
gp_htdkty_sector

GICS3_name           fiscalDate
Bán lẻ               2020-03-31     9082.767226
                     2020-06-30     8067.325921
                     2020-09-30     8511.420839
                     2020-12-31     9926.679061
                     2021-03-31    10661.732208
                                       ...     
Đồ gia dụng lâu bền  2023-12-31     1052.914516
                     2024-03-31      876.590519
                     2024-06-30      777.235020
                     2024-09-30      763.065782
                     2024-12-31      936.723965
Name: Lợi nhuận gộp, Length: 140, dtype: float64

In [83]:
gm_htdkty_sector = gp_htdkty_sector/revni_htdkty_sector[REVENUE]
gm_htdkty_sector

GICS3_name           fiscalDate
Bán lẻ               2020-03-31    0.141659
                     2020-06-30    0.144482
                     2020-09-30    0.137748
                     2020-12-31    0.142874
                     2021-03-31    0.149385
                                     ...   
Đồ gia dụng lâu bền  2023-12-31    0.151126
                     2024-03-31    0.163385
                     2024-06-30    0.135068
                     2024-09-30    0.164265
                     2024-12-31    0.137799
Length: 140, dtype: float64

In [84]:
current_gm_htdkty_sector = gm_htdkty_sector.loc(axis=0)[:, CURRENT_DATE]
current_gm_htdkty_sector

GICS3_name                          fiscalDate
Bán lẻ                              2024-12-31    0.137664
Dệt may & Trang phục, phụ kiện      2024-12-31    0.147226
Khách sạn, nhà hàng & Giải trí      2024-12-31    0.189844
Lốp & Cao su                        2024-12-31    0.304275
Nhà sản xuất ô tô và thiết bị ô tô  2024-12-31    0.047577
Truyền thông và xuất bản            2024-12-31    0.228881
Đồ gia dụng lâu bền                 2024-12-31    0.137799
dtype: float64

In [85]:
current_growth_rate_htdkty_sector = growth_rate_htdkty_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_htdkty_sector.index = (
    current_growth_rate_htdkty_sector.index.get_level_values(GICS3)
)
current_growth_rate_htdkty_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Bán lẻ,0.093021,1.170977
"Dệt may & Trang phục, phụ kiện",0.142237,1.079325
"Khách sạn, nhà hàng & Giải trí",0.221651,12.356954
Lốp & Cao su,0.145116,0.614673
Nhà sản xuất ô tô và thiết bị ô tô,0.216220,-1.057641
Truyền thông và xuất bản,0.267899,0.420614
Đồ gia dụng lâu bền,-0.024311,-0.382926


In [86]:
current_htdkty_summary: pd.DataFrame = current_growth_rate_htdkty_sector.rename(
    columns={REVENUE: REV_GROWTH, NI: NI_GROWTH}
).join(current_ni_htdkty_sector)
current_htdkty_summary


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,,
Bán lẻ,0.093021,1.170977,2147.671238
"Dệt may & Trang phục, phụ kiện",0.142237,1.079325,1107.777883
"Khách sạn, nhà hàng & Giải trí",0.221651,12.356954,311.480353
Lốp & Cao su,0.145116,0.614673,3384.518137
Nhà sản xuất ô tô và thiết bị ô tô,0.216220,-1.057641,-10.206478
Truyền thông và xuất bản,0.267899,0.420614,237.692061
Đồ gia dụng lâu bền,-0.024311,-0.382926,125.542059


In [87]:
fig3 = px.scatter(
    current_htdkty_summary,
    x=REV_GROWTH,
    y=NI_GROWTH,
    color=current_htdkty_summary.index,
)
fig3

## 4.3. <a id='toc4_3_'></a>[Bán lẻ](#toc0_)

In [88]:
bl_pl = (
    htdkty_pl[htdkty_pl[GICS3] == "Bán lẻ"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
bl_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
ABS,2020-03-31,112.752409,0.005249,112.747160,104.640398,8.106762,0.287103,1.948921,0.000000,0.000000,2.595584,...,0.405887,0.000000,1.623548,0.000000,1.623548,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
AME,2020-03-31,255.087192,0.000000,255.087192,234.182248,20.904944,0.029859,12.641983,12.641983,0.000000,0.000000,...,0.114871,0.000000,0.375073,0.000000,0.375073,1.500000e-08,1.500000e-08,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
BTT,2020-03-31,60.502210,0.000000,60.502210,34.032123,26.470087,1.754099,1.593219,0.000000,-1.465826,5.061546,...,2.747235,-0.575084,8.473863,-0.026258,8.500121,5.760000e-07,NaN,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
CCI,2020-03-31,85.911799,0.000000,85.911799,75.560030,10.351769,5.361153,-0.096458,0.000000,0.000000,2.566909,...,2.160925,0.000000,8.459062,0.000000,8.459062,4.820000e-07,0.000000e+00,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
CDP,2020-03-31,731.856412,11.432058,720.424354,676.037482,44.386873,2.913545,10.222241,9.806820,0.000000,23.827051,...,1.060249,0.000000,4.252147,0.000000,4.252147,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TMC,2024-12-31,609.848731,0.036215,609.812516,573.799002,36.013513,1.396130,0.248552,0.000000,0.000000,24.518764,...,0.863637,0.000000,2.929621,0.000000,2.929621,2.360000e-07,2.360000e-07,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
TNA,2024-12-31,528.598028,0.000000,528.598028,522.965574,5.632454,-1.193556,10.611794,10.602380,0.000000,2.499831,...,0.000000,0.000000,-25.129964,-0.126523,-25.003442,-5.080000e-07,-5.080000e-07,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ
TTH,2024-12-31,214.551375,0.000000,214.551375,209.592188,4.959187,0.000247,0.000000,0.000000,0.000000,0.661200,...,0.164452,0.000000,3.395228,0.000000,3.395228,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Bán lẻ,Bán lẻ


In [89]:
revni_bl = bl_pl[[REVENUE, NI]]
revni_bl

,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
ABS,2020-03-31,112.747160,1.623548
AME,2020-03-31,255.087192,0.375073
BTT,2020-03-31,60.502210,8.473863
CCI,2020-03-31,85.911799,8.459062
CDP,2020-03-31,720.424354,4.252147
...,...,...,...
TMC,2024-12-31,609.812516,2.929621
TNA,2024-12-31,528.598028,-25.129964
TTH,2024-12-31,214.551375,3.395228


In [ ]:
top_DN_1 = revni_bl.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=NI,ascending=False).head(10).index
top_DN_1

Index(['MWG', 'PNJ', 'DGW', 'FRT', 'SVC', 'PET', 'HAX', 'CLX', 'PCT', 'HID'], dtype='object', name='Ticker')

In [ ]:
bl_current_growth = current_yoy_growth(revni_bl)
bl_current_growth = bl_current_growth.sort_values(by=NI, ascending=False)
bl_current_growth_topDN = bl_current_growth[bl_current_growth.index.isin(top_DN_1)]
bl_current_growth_topDN

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
SVC,0.326934,12.807331
MWG,0.100327,8.434928
HAX,0.687782,1.648070
DGW,0.208354,0.623183
PCT,-0.012699,0.559436
PET,0.030025,0.457714
PNJ,-0.120798,0.159117
CLX,-0.011411,0.100136
FRT,0.317334,-2.293440


## 4.4. <a id='toc4_4_'></a>[Dệt may & Trang phục, phụ kiện](#toc0_)

In [92]:
dm_pl = (
    htdkty_pl[htdkty_pl[GICS3] == "Dệt may & Trang phục, phụ kiện"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
dm_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
AAT,2020-03-31,45.982506,0.000000,45.982506,38.706874,7.275633,0.034282,3.036461,3.013552,0.000000,0.845948,...,2.053040,0.000000,8.195431,0.000000,8.195431,2.360000e-07,2.360000e-07,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
ADS,2020-03-31,294.997206,0.000000,294.997206,284.960291,10.036915,8.694010,10.741577,10.741577,0.000000,2.293477,...,0.129660,0.000000,-2.108492,0.279151,-2.387642,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
BDG,2020-03-31,320.716049,0.000000,320.716049,256.170651,64.545399,3.295892,2.976743,0.416760,-2.319700,5.788362,...,6.596943,0.197687,24.045383,0.000198,24.045185,2.004000e-06,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
DM7,2020-03-31,261.246352,0.000000,261.246352,214.460640,46.785712,1.764706,0.000000,0.000000,0.000000,10.559002,...,3.665960,0.000000,14.663842,0.000000,14.663842,0.000000e+00,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
EVE,2020-03-31,187.853018,0.097718,187.755299,138.452352,49.302948,12.808528,11.512598,4.353567,0.000000,37.121051,...,0.000000,-0.119829,-11.787548,0.000000,-11.787548,-3.100000e-07,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TNG,2024-12-31,1851.566066,0.000000,1851.566066,1575.785170,275.780896,35.066052,89.252610,55.018858,0.000000,21.693704,...,15.834643,0.000000,74.567169,0.000000,74.567169,6.080000e-07,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
TVT,2024-12-31,450.731332,0.002038,450.729294,384.126366,66.602928,2.933143,6.824195,6.793432,0.008822,6.559324,...,4.943221,1.118989,12.801181,-0.479419,13.280600,6.150000e-07,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"
VGG,2024-12-31,2130.452324,0.917380,2129.534944,1853.972806,275.562138,31.128383,7.068277,0.274094,26.622343,130.486948,...,13.820750,0.000000,100.974801,6.520557,94.454244,1.755000e-06,0.000000e+00,Hàng tiêu dùng không thiết yếu,Hàng tiêu dùng lâu bền và trang phục,"Dệt may & Trang phục, phụ kiện"


In [93]:
revni_dm = dm_pl[[REVENUE, NI]]
revni_dm

,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
AAT,2020-03-31,45.982506,8.195431
ADS,2020-03-31,294.997206,-2.108492
BDG,2020-03-31,320.716049,24.045383
DM7,2020-03-31,261.246352,14.663842
EVE,2020-03-31,187.755299,-11.787548
...,...,...,...
TNG,2024-12-31,1851.566066,74.567169
TVT,2024-12-31,450.729294,12.801181
VGG,2024-12-31,2129.534944,100.974801


In [94]:
revni_dm.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=REVENUE,ascending=False)

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
VGT,4819.290362,278.652446
VGG,2129.534944,100.974801
TNG,1851.566066,74.567169
MSH,1428.376237,170.353720
MNB,1401.109778,47.102710
HTG,1331.167067,94.002182
M10,1306.379221,19.843244
TCM,924.982593,61.651368
X20,575.541856,29.766422


In [ ]:
top_DN_2 = revni_dm.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=NI,ascending=False).head(10).index
top_DN_2

Index(['VGT', 'MSH', 'PPH', 'VGG', 'HTG', 'BDG', 'TNG', 'TCM', 'MNB', 'ADS'], dtype='object', name='Ticker')

In [ ]:
dm_current_growth = current_yoy_growth(revni_dm)
dm_current_growth = dm_current_growth.sort_values(by=NI, ascending=False)
dm_current_growth_topDN = dm_current_growth[dm_current_growth.index.isin(top_DN_2)]
dm_current_growth_topDN

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
MNB,0.396977,10.219198
HTG,0.188126,2.188632
TCM,0.135504,1.757127
ADS,0.300581,1.449596
MSH,0.234886,1.094458
VGT,0.120737,1.055159
PPH,0.071322,0.900385
VGG,-0.039634,0.804725
BDG,0.303947,0.351374


# 5. <a id='toc5_'></a>[Hàng tiêu dùng thiết yếu](#toc0_)

## 5.1. <a id='toc5_1_'></a>[Toàn ngành](#toc0_)

In [97]:
revni_htdty = revni_sector.loc["Hàng tiêu dùng thiết yếu"]
revni_htdty

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,87513.278968,5748.511839
2020-06-30,97248.242489,8291.559040
2020-09-30,106757.421435,9581.088084
2020-12-31,110693.067033,8371.990905
2021-03-31,93353.237997,7277.086484
2021-06-30,108970.162166,9731.893057
2021-09-30,103313.312738,9202.958248
2021-12-31,123108.169657,16789.454605
2022-03-31,101425.188794,10580.449575


In [98]:
cagr(revni_htdty)

Doanh thu thuần                             0.075573
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.122010
dtype: float64

In [99]:
nim_htdty = revni_htdty[NI] / revni_htdty[REVENUE]
nim_htdty

fiscalDate
2020-03-31    0.065687
2020-06-30    0.085262
2020-09-30    0.089746
2020-12-31    0.075632
2021-03-31    0.077952
2021-06-30    0.089308
2021-09-30    0.089078
2021-12-31    0.136380
2022-03-31    0.104318
2022-06-30    0.090919
2022-09-30    0.082395
2022-12-31    0.044828
2023-03-31    0.071091
2023-06-30    0.082662
2023-09-30    0.077982
2023-12-31    0.073570
2024-03-31    0.075518
2024-06-30    0.084285
2024-09-30    0.087126
2024-12-31    0.081146
dtype: float64

In [100]:
gp_htdty = ptcpl.loc[ptcpl[GICS1] == "Hàng tiêu dùng thiết yếu", "Lợi nhuận gộp"].groupby(level=0).sum()
gp_htdty

fiscalDate
2020-03-31    21416.522583
2020-06-30    24235.511596
2020-09-30    26806.335477
2020-12-31    26920.630412
2021-03-31    21905.426781
2021-06-30    25758.947027
2021-09-30    26075.597889
2021-12-31    30370.741251
2022-03-31    25183.983658
2022-06-30    28647.738601
2022-09-30    28400.814712
2022-12-31    27169.327007
2023-03-31    22229.949096
2023-06-30    27235.783009
2023-09-30    28141.307841
2023-12-31    28464.586125
2024-03-31    25583.607314
2024-06-30    29982.794711
2024-09-30    30670.506741
2024-12-31    31426.190869
Name: Lợi nhuận gộp, dtype: float64

In [101]:
gm_htdty = gp_htdty / revni_htdty[REVENUE]
gm_htdty

fiscalDate
2020-03-31    0.244723
2020-06-30    0.249213
2020-09-30    0.251096
2020-12-31    0.243201
2021-03-31    0.234651
2021-06-30    0.236385
2021-09-30    0.252393
2021-12-31    0.246700
2022-03-31    0.248301
2022-06-30    0.248522
2022-09-30    0.240729
2022-12-31    0.227572
2023-03-31    0.226843
2023-06-30    0.243418
2023-09-30    0.248258
2023-12-31    0.238700
2024-03-31    0.241964
2024-06-30    0.243625
2024-09-30    0.252421
2024-12-31    0.249471
dtype: float64

## 5.2. <a id='toc5_2_'></a>[Ngành cấp 3](#toc0_)

In [102]:
htdty_pl = ptcpl[ptcpl[GICS1] == "Hàng tiêu dùng thiết yếu"]
revni_htdty_sector = htdty_pl.groupby([GICS3, htdty_pl.index])[[REVENUE, NI]].sum()
revni_htdty_sector

Doanh thu thuần  \
GICS3_name                      fiscalDate                    
Sản phẩm gia dụng không lâu bền 2020-03-31      2022.364216   
                                2020-06-30      2227.021348   
                                2020-09-30      2476.863817   
                                2020-12-31      2243.921296   
                                2021-03-31      2265.535621   
...                                                     ...   
Đồ uống                         2023-12-31     14401.905581   
                                2024-03-31     11361.000998   
                                2024-06-30     14164.288692   
                                2024-09-30     13419.581978   
                                2024-12-31     14870.386253   

                                            Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name                      fiscalDate                                            
Sản phẩm gia dụng không lâu bền 2020-03-31                                101.131362  
                                2020-06-30                                162.840642  
                                2020-09-30                                252.578055  
                                2020-12-31                                253.298267  
                                2021-03-31                                193.904054  
...                                                                              ...  
Đồ uống                         2023-12-31                               1158.897606  
                                2024-03-31                               1115.099300  
                                2024-06-30                               1714.819653  
                                2024-09-30                               1446.863883  
                                2024-12-31                               1262.763697  

[80 rows x 2 columns]

In [103]:
current_ni_htdty_sector = revni_htdty_sector.loc(axis=0)[:, CURRENT_DATE][NI]
current_ni_htdty_sector.index = current_ni_htdty_sector.index.get_level_values(GICS3)
current_ni_htdty_sector

GICS3_name
Sản phẩm gia dụng không lâu bền     288.913680
Thuốc lá                             32.931809
Thực phẩm                          8637.488630
Đồ uống                            1262.763697
Name: Lợi nhuận sau thuế thu nhập doanh nghiệp, dtype: float64

In [104]:
#revni_htdkty_sector.xs("Khách sạn, nhà hàng & Giải trí", level=GICS3)

In [105]:
# htdkty_pl[htdkty_pl.index.isin(["2023-12-31", "2024-12-31"])][["Ticker",NI]].sort_values(by=NI, ascending=False)

In [106]:
growth_rate_htdty_sector = revni_htdty_sector.groupby(level=0).pct_change(4)
growth_rate_htdty_sector

Doanh thu thuần  \
GICS3_name                      fiscalDate                    
Sản phẩm gia dụng không lâu bền 2020-03-31              NaN   
                                2020-06-30              NaN   
                                2020-09-30              NaN   
                                2020-12-31              NaN   
                                2021-03-31         0.120241   
...                                                     ...   
Đồ uống                         2023-12-31        -0.099724   
                                2024-03-31         0.115356   
                                2024-06-30         0.019760   
                                2024-09-30         0.028429   
                                2024-12-31         0.032529   

                                            Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name                      fiscalDate                                            
Sản phẩm gia dụng không lâu bền 2020-03-31                                       NaN  
                                2020-06-30                                       NaN  
                                2020-09-30                                       NaN  
                                2020-12-31                                       NaN  
                                2021-03-31                                  0.917348  
...                                                                              ...  
Đồ uống                         2023-12-31                                 -0.072335  
                                2024-03-31                                 -0.012361  
                                2024-06-30                                  0.059924  
                                2024-09-30                                  0.077535  
                                2024-12-31                                  0.089625  

[80 rows x 2 columns]

In [107]:
current_growth_rate_htdty_sector = growth_rate_htdty_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_htdty_sector.index = (
    current_growth_rate_htdty_sector.index.get_level_values(GICS3)
)
current_growth_rate_htdty_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Sản phẩm gia dụng không lâu bền,-0.093444,0.190466
Thuốc lá,0.246062,-0.326103
Thực phẩm,0.062184,0.179558
Đồ uống,0.032529,0.089625


In [108]:
gp_htdty_sector = htdty_pl.groupby([GICS3, htdty_pl.index])["Lợi nhuận gộp"].sum()
gp_htdty_sector

GICS3_name                       fiscalDate
Sản phẩm gia dụng không lâu bền  2020-03-31     616.338573
                                 2020-06-30     594.883645
                                 2020-09-30     787.858083
                                 2020-12-31     676.598054
                                 2021-03-31     617.430136
                                                  ...     
Đồ uống                          2023-12-31    3551.860979
                                 2024-03-31    2806.381154
                                 2024-06-30    3723.155720
                                 2024-09-30    3442.171665
                                 2024-12-31    3697.105975
Name: Lợi nhuận gộp, Length: 80, dtype: float64

In [109]:
gm_htdty_sector = gp_htdty_sector/revni_htdty_sector[REVENUE]
gm_htdty_sector

GICS3_name                       fiscalDate
Sản phẩm gia dụng không lâu bền  2020-03-31    0.304761
                                 2020-06-30    0.267121
                                 2020-09-30    0.318087
                                 2020-12-31    0.301525
                                 2021-03-31    0.272532
                                                 ...   
Đồ uống                          2023-12-31    0.246624
                                 2024-03-31    0.247019
                                 2024-06-30    0.262855
                                 2024-09-30    0.256504
                                 2024-12-31    0.248622
Length: 80, dtype: float64

In [110]:
current_gm_htdty_sector = gm_htdty_sector.loc(axis=0)[:, CURRENT_DATE]
current_gm_htdty_sector

GICS3_name                       fiscalDate
Sản phẩm gia dụng không lâu bền  2024-12-31    0.294264
Thuốc lá                         2024-12-31    0.105150
Thực phẩm                        2024-12-31    0.250417
Đồ uống                          2024-12-31    0.248622
dtype: float64

In [111]:
current_growth_rate_htdty_sector = growth_rate_htdty_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_htdty_sector.index = (
    current_growth_rate_htdty_sector.index.get_level_values(GICS3)
)
current_growth_rate_htdty_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Sản phẩm gia dụng không lâu bền,-0.093444,0.190466
Thuốc lá,0.246062,-0.326103
Thực phẩm,0.062184,0.179558
Đồ uống,0.032529,0.089625


In [112]:
current_htdty_summary: pd.DataFrame = current_growth_rate_htdty_sector.rename(
    columns={REVENUE: REV_GROWTH, NI: NI_GROWTH}
).join(current_ni_htdty_sector)
current_htdty_summary


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,,
Sản phẩm gia dụng không lâu bền,-0.093444,0.190466,288.913680
Thuốc lá,0.246062,-0.326103,32.931809
Thực phẩm,0.062184,0.179558,8637.488630
Đồ uống,0.032529,0.089625,1262.763697


In [113]:
px.scatter(
    current_htdty_summary,
    x=REV_GROWTH,
    y=NI_GROWTH,
    color=current_htdty_summary.index,
)

## 5.3. <a id='toc5_3_'></a>[Thực phẩm](#toc0_)

In [114]:
tp_pl = (
    htdty_pl[htdty_pl[GICS3] == "Thực phẩm"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
tp_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
AAM,2020-03-31,41.176955,0.000000,41.176955,36.136953,5.040002,0.647222,0.009707,0.008899,0.000000,2.948184,...,0.089671,0.000000,0.665805,0.000000,0.665805,5.300000e-08,5.300000e-08,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
ABT,2020-03-31,75.347063,0.000000,75.347063,67.382882,7.964181,1.087358,1.156505,0.972148,0.000000,4.136712,...,0.088923,0.000000,0.711124,0.000000,0.712897,6.200000e-08,0.000000e+00,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
ACL,2020-03-31,237.392033,3.095341,234.296692,205.046060,29.250632,1.928084,10.169436,10.159150,0.000000,12.974542,...,0.155909,-0.015576,1.053230,0.000000,1.053230,4.600000e-08,4.600000e-08,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
AFX,2020-03-31,174.362907,2.037493,172.325414,163.289386,9.036028,0.002902,1.833520,1.442601,0.000000,3.780702,...,-0.000232,0.000000,0.349901,0.000000,0.349901,1.000000e-08,0.000000e+00,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
AGF,2020-03-31,173.342102,1.052826,172.289276,151.031305,21.257971,0.325021,10.556320,10.537275,0.000000,9.176990,...,0.000000,0.000000,-2.090245,0.000000,-2.090245,-7.400000e-08,-7.400000e-08,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VHF,2024-12-31,165.740785,2.695463,163.045322,159.855546,3.189776,2.465651,1.016723,0.706764,0.000000,3.173444,...,0.000000,0.000000,-0.971995,0.000000,-0.971995,0.000000e+00,0.000000e+00,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
VLC,2024-12-31,759.420773,0.953641,758.467131,564.238961,194.228171,49.252200,1.329297,1.158042,2.146467,159.121623,...,9.616333,-3.578643,47.011816,21.765710,25.246106,9.900000e-08,0.000000e+00,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm
VNM,2024-12-31,15485.051437,7.978311,15477.073125,9267.382480,6209.690645,394.631908,140.107173,66.109862,23.396010,3351.121539,...,555.142311,-58.564884,2146.791344,23.142075,2123.649268,9.080000e-07,0.000000e+00,Hàng tiêu dùng thiết yếu,Thực phẩm & Đồ uống,Thực phẩm


In [115]:
revni_tp = tp_pl[[REVENUE, NI]]
revni_tp

,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
AAM,2020-03-31,41.176955,0.665805
ABT,2020-03-31,75.347063,0.711124
ACL,2020-03-31,234.296692,1.053230
AFX,2020-03-31,172.325414,0.349901
AGF,2020-03-31,172.289276,-2.090245
...,...,...,...
VHF,2024-12-31,163.045322,-0.971995
VLC,2024-12-31,758.467131,47.011816
VNM,2024-12-31,15477.073125,2146.791344


In [ ]:
top_DN_3 = revni_tp.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=NI,ascending=False).head(10).index
top_DN_3

Index(['MCH', 'VNM', 'MSN', 'QNS', 'VHC', 'PAN', 'DBC', 'SBT', 'HAG', 'TID'], dtype='object', name='Ticker')

In [ ]:
tp_current_growth = current_yoy_growth(revni_tp)
tp_current_growth = tp_current_growth.sort_values(by=NI, ascending=False)
tp_current_growth_topDN = tp_current_growth[tp_current_growth.index.isin(top_DN_3)]
tp_current_growth_topDN

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
DBC,0.381507,36.020329
VHC,0.338171,5.641200
MSN,0.092369,1.992705
TID,0.405232,0.779320
SBT,0.074050,0.301732
PAN,0.016720,0.176964
MCH,0.052845,0.026767
QNS,-0.043863,-0.048515
VNM,-0.009068,-0.086746


# 6. <a id='toc6_'></a>[Nguyên vật liệu](#toc0_)

## 6.1. <a id='toc6_1_'></a>[Toàn ngành](#toc0_)

In [205]:
revni_nvl = revni_sector.loc["Nguyên vật liệu"]
revni_nvl

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
fiscalDate,,
2020-03-31,90767.909244,4145.860020
2020-06-30,98575.388837,6196.674675
2020-09-30,105643.200479,7622.783925
2020-12-31,119740.898839,9496.737064
2021-03-31,132610.382500,12576.129170
2021-06-30,148760.889397,18659.016666
2021-09-30,138824.971469,16196.144895
2021-12-31,177371.861063,17070.525293
2022-03-31,172235.932937,19183.969961


In [206]:
cagr(revni_nvl)

Doanh thu thuần                             0.102278
Lợi nhuận sau thuế thu nhập doanh nghiệp    0.106112
dtype: float64

In [207]:
nim_nvl = revni_nvl[NI] / revni_nvl[REVENUE]
nim_nvl

fiscalDate
2020-03-31    0.045675
2020-06-30    0.062862
2020-09-30    0.072156
2020-12-31    0.079311
2021-03-31    0.094835
2021-06-30    0.125430
2021-09-30    0.116666
2021-12-31    0.096241
2022-03-31    0.111382
2022-06-30    0.081492
2022-09-30    0.011937
2022-12-31    0.001896
2023-03-31    0.027811
2023-06-30    0.027895
2023-09-30    0.036703
2023-12-31    0.050164
2024-03-31    0.047566
2024-06-30    0.055451
2024-09-30    0.040534
2024-12-31    0.046475
dtype: float64

In [208]:
gp_nvl = ptcpl.loc[ptcpl[GICS1] == "Nguyên vật liệu", "Lợi nhuận gộp"].groupby(level=0).sum()
gp_nvl

fiscalDate
2020-03-31    12780.172614
2020-06-30    14123.871940
2020-09-30    16036.076471
2020-12-31    18766.613870
2021-03-31    22316.145074
2021-06-30    29746.493606
2021-09-30    26223.035022
2021-12-31    29342.704837
2022-03-31    31082.655522
2022-06-30    26899.940941
2022-09-30    13386.606569
2022-12-31    11457.092146
2023-03-31    12794.954261
2023-06-30    14113.029677
2023-09-30    14388.680578
2023-12-31    15716.017421
2024-03-31    14910.284570
2024-06-30    18912.717122
2024-09-30    16173.348360
2024-12-31    18437.071755
Name: Lợi nhuận gộp, dtype: float64

In [209]:
gm_nvl = gp_nvl / revni_nvl[REVENUE]
gm_nvl

fiscalDate
2020-03-31    0.140801
2020-06-30    0.143280
2020-09-30    0.151795
2020-12-31    0.156727
2021-03-31    0.168284
2021-06-30    0.199962
2021-09-30    0.188893
2021-12-31    0.165430
2022-03-31    0.180466
2022-06-30    0.163596
2022-09-30    0.094220
2022-12-31    0.083803
2023-03-31    0.104927
2023-06-30    0.109974
2023-09-30    0.114295
2023-12-31    0.115101
2024-03-31    0.120955
2024-06-30    0.126228
2024-09-30    0.119981
2024-12-31    0.124826
dtype: float64

## 6.2. <a id='toc6_2_'></a>[Ngành cấp 3](#toc0_)

In [210]:
nvl_pl = ptcpl[ptcpl[GICS1] == "Nguyên vật liệu"]
revni_nvl_sector = nvl_pl.groupby([GICS3, nvl_pl.index])[[REVENUE, NI]].sum()
revni_nvl_sector

Doanh thu thuần  \
GICS3_name         fiscalDate                    
Bao bì và đóng gói 2020-03-31      5670.491230   
                   2020-06-30      5426.036596   
                   2020-09-30      5996.777604   
                   2020-12-31      6758.203121   
                   2021-03-31      6710.237589   
...                                        ...   
Vật liệu xây dựng  2023-12-31     20474.070228   
                   2024-03-31     15746.837391   
                   2024-06-30     19917.883699   
                   2024-09-30     18449.669990   
                   2024-12-31     22618.733678   

                               Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name         fiscalDate                                            
Bao bì và đóng gói 2020-03-31                                230.844547  
                   2020-06-30                                301.973787  
                   2020-09-30                                280.876909  
                   2020-12-31                                328.714515  
                   2021-03-31                                328.811158  
...                                                                 ...  
Vật liệu xây dựng  2023-12-31                                760.503380  
                   2024-03-31                                600.637073  
                   2024-06-30                               1325.924348  
                   2024-09-30                               1119.757227  
                   2024-12-31                               1593.142338  

[160 rows x 2 columns]

In [211]:
current_ni_nvl_sector = revni_nvl_sector.loc(axis=0)[:, CURRENT_DATE][NI]
current_ni_nvl_sector.index = current_ni_nvl_sector.index.get_level_values(GICS3)
current_ni_nvl_sector

GICS3_name
Bao bì và đóng gói           254.119114
Giấy                          16.397782
Gỗ và các sản phẩm từ gỗ     351.260459
Hóa chất                     929.961323
Kim loại không chứa sắt      339.101609
Phân bón                     635.644396
Thép                        2744.886506
Vật liệu xây dựng           1593.142338
Name: Lợi nhuận sau thuế thu nhập doanh nghiệp, dtype: float64

In [212]:
#revni_htdkty_sector.xs("Khách sạn, nhà hàng & Giải trí", level=GICS3)

In [213]:
# htdkty_pl[htdkty_pl.index.isin(["2023-12-31", "2024-12-31"])][["Ticker",NI]].sort_values(by=NI, ascending=False)

In [214]:
growth_rate_nvl_sector = revni_nvl_sector.groupby(level=0).pct_change(4)
growth_rate_nvl_sector

Doanh thu thuần  \
GICS3_name         fiscalDate                    
Bao bì và đóng gói 2020-03-31              NaN   
                   2020-06-30              NaN   
                   2020-09-30              NaN   
                   2020-12-31              NaN   
                   2021-03-31         0.183361   
...                                        ...   
Vật liệu xây dựng  2023-12-31        -0.114780   
                   2024-03-31        -0.129440   
                   2024-06-30        -0.049120   
                   2024-09-30        -0.000447   
                   2024-12-31         0.104750   

                               Lợi nhuận sau thuế thu nhập doanh nghiệp  
GICS3_name         fiscalDate                                            
Bao bì và đóng gói 2020-03-31                                       NaN  
                   2020-06-30                                       NaN  
                   2020-09-30                                       NaN  
                   2020-12-31                                       NaN  
                   2021-03-31                                  0.424383  
...                                                                 ...  
Vật liệu xây dựng  2023-12-31                                 -0.388207  
                   2024-03-31                                 -0.225443  
                   2024-06-30                                 -0.192732  
                   2024-09-30                                 -0.083307  
                   2024-12-31                                  1.094852  

[160 rows x 2 columns]

In [215]:
current_growth_rate_nvl_sector = growth_rate_nvl_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_nvl_sector.index = (
    current_growth_rate_nvl_sector.index.get_level_values(GICS3)
)
current_growth_rate_nvl_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Bao bì và đóng gói,0.189996,-0.141639
Giấy,0.258311,-0.149858
Gỗ và các sản phẩm từ gỗ,0.108739,0.042636
Hóa chất,0.109864,-0.004110
Kim loại không chứa sắt,0.249273,-1.445437
Phân bón,0.136565,-0.743263
Thép,0.021112,-0.015598
Vật liệu xây dựng,0.104750,1.094852


In [216]:
gp_nvl_sector = nvl_pl.groupby([GICS3, nvl_pl.index])["Lợi nhuận gộp"].sum()
gp_nvl_sector

GICS3_name          fiscalDate
Bao bì và đóng gói  2020-03-31     786.348188
                    2020-06-30     751.369382
                    2020-09-30     847.630662
                    2020-12-31     945.969649
                    2021-03-31     982.432693
                                     ...     
Vật liệu xây dựng   2023-12-31    3398.523214
                    2024-03-31    2664.609743
                    2024-06-30    3646.495578
                    2024-09-30    3466.145028
                    2024-12-31    4339.683135
Name: Lợi nhuận gộp, Length: 160, dtype: float64

In [217]:
gm_nvl_sector = gp_nvl_sector/revni_nvl_sector[REVENUE]
gm_nvl_sector

GICS3_name          fiscalDate
Bao bì và đóng gói  2020-03-31    0.138674
                    2020-06-30    0.138475
                    2020-09-30    0.141348
                    2020-12-31    0.139974
                    2021-03-31    0.146408
                                    ...   
Vật liệu xây dựng   2023-12-31    0.165992
                    2024-03-31    0.169216
                    2024-06-30    0.183076
                    2024-09-30    0.187870
                    2024-12-31    0.191862
Length: 160, dtype: float64

In [218]:
current_gm_nvl_sector = gm_nvl_sector.loc(axis=0)[:, CURRENT_DATE]
current_gm_nvl_sector

GICS3_name                fiscalDate
Bao bì và đóng gói        2024-12-31    0.121150
Giấy                      2024-12-31    0.066654
Gỗ và các sản phẩm từ gỗ  2024-12-31    0.183019
Hóa chất                  2024-12-31    0.194375
Kim loại không chứa sắt   2024-12-31    0.132179
Phân bón                  2024-12-31    0.133692
Thép                      2024-12-31    0.089194
Vật liệu xây dựng         2024-12-31    0.191862
dtype: float64

In [219]:
current_growth_rate_nvl_sector = growth_rate_nvl_sector.loc(axis=0)[:, CURRENT_DATE]
current_growth_rate_nvl_sector.index = (
    current_growth_rate_nvl_sector.index.get_level_values(GICS3)
)
current_growth_rate_nvl_sector


,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,
Bao bì và đóng gói,0.189996,-0.141639
Giấy,0.258311,-0.149858
Gỗ và các sản phẩm từ gỗ,0.108739,0.042636
Hóa chất,0.109864,-0.004110
Kim loại không chứa sắt,0.249273,-1.445437
Phân bón,0.136565,-0.743263
Thép,0.021112,-0.015598
Vật liệu xây dựng,0.104750,1.094852


In [220]:
current_nvl_summary: pd.DataFrame = current_growth_rate_nvl_sector.rename(
    columns={REVENUE: REV_GROWTH, NI: NI_GROWTH}
).join(current_ni_nvl_sector)
current_nvl_summary


,Tăng trưởng doanh thu thuần YoY,Tăng trưởng LNST YoY,Lợi nhuận sau thuế thu nhập doanh nghiệp
GICS3_name,,,
Bao bì và đóng gói,0.189996,-0.141639,254.119114
Giấy,0.258311,-0.149858,16.397782
Gỗ và các sản phẩm từ gỗ,0.108739,0.042636,351.260459
Hóa chất,0.109864,-0.004110,929.961323
Kim loại không chứa sắt,0.249273,-1.445437,339.101609
Phân bón,0.136565,-0.743263,635.644396
Thép,0.021112,-0.015598,2744.886506
Vật liệu xây dựng,0.104750,1.094852,1593.142338


In [221]:
px.scatter(
    current_nvl_summary,
    x=REV_GROWTH,
    y=NI_GROWTH, 
    size = NI,
    color=current_nvl_summary.index,
)

## 6.3. <a id='toc6_3_'></a>[Vật liệu xây dựng](#toc0_)

In [222]:
vlxd_pl = (
    nvl_pl[nvl_pl[GICS3] == "Vật liệu xây dựng"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
vlxd_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
AMC,2020-03-31,31.040374,0.000000,31.040374,17.103877,13.936498,0.029073,0.363479,0.362342,0.000000,10.120540,...,0.061746,0.000000,1.139044,0.000000,1.139044,3.330000e-07,3.330000e-07,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BCC,2020-03-31,1050.945519,0.000000,1050.945519,921.928505,129.017014,0.014527,20.872340,20.872340,0.000000,40.125413,...,6.507216,0.000000,18.575534,-1.615877,20.191411,1.840000e-07,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BDT,2020-03-31,109.083946,0.000000,109.083946,72.451059,36.632887,0.001538,2.858451,2.858451,0.000000,8.098063,...,4.021335,0.000000,16.092861,0.000000,16.092861,3.940000e-07,NaN,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BMP,2020-03-31,1020.368268,0.588220,1019.780048,774.099922,245.680126,17.505577,25.859785,0.009238,-0.116213,88.460624,...,25.284308,0.336890,102.368580,0.000000,102.368580,1.251000e-06,1.251000e-06,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BTS,2020-03-31,688.661507,0.000000,688.661507,599.310724,89.350783,1.737259,22.433319,21.766638,0.000000,26.120832,...,3.938214,0.000000,11.447745,0.000000,11.447745,9.500000e-08,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VHL,2024-12-31,335.499383,0.000000,335.499383,296.285124,39.214260,0.985491,0.516961,0.468762,-3.625017,30.951928,...,1.153334,0.794371,-15.593147,0.000000,-15.593147,-6.280000e-07,-6.280000e-07,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
VIT,2024-12-31,707.806833,4.316474,703.490359,633.758583,69.731776,0.103792,23.226103,21.909737,0.000000,1.801414,...,7.537723,0.000000,35.292465,0.000000,35.292465,7.060000e-07,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
VLB,2024-12-31,358.973686,0.000000,358.973686,265.978228,92.995458,12.609517,0.000000,0.000000,0.000000,3.745993,...,17.116436,-1.420922,62.635058,0.000000,62.635058,1.140000e-06,1.140000e-06,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng


In [223]:
revni_vlxd = vlxd_pl[[REVENUE, NI]]
revni_vlxd

,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
AMC,2020-03-31,31.040374,1.139044
BCC,2020-03-31,1050.945519,18.575534
BDT,2020-03-31,109.083946,16.092861
BMP,2020-03-31,1019.780048,102.368580
BTS,2020-03-31,688.661507,11.447745
...,...,...,...
VHL,2024-12-31,335.499383,-15.593147
VIT,2024-12-31,703.490359,35.292465
VLB,2024-12-31,358.973686,62.635058


In [ ]:
top_DN_4 = revni_vlxd.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=NI,ascending=False).head(10).index
top_DN_4

Index(['VGC', 'BMP', 'NTP', 'VCS', 'LIC', 'VLB', 'THG', 'BCC', 'VIT', 'TDF'], dtype='object', name='Ticker')

In [ ]:
vlxd_current_growth = current_yoy_growth(revni_vlxd)
vlxd_current_growth = vlxd_current_growth.sort_values(by=NI, ascending=False)
vlxd_current_growth_topDN = vlxd_current_growth[vlxd_current_growth.index.isin(top_DN_4)]
vlxd_current_growth_topDN

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
VIT,0.253010,3.112319
LIC,0.515811,2.917338
THG,0.403528,1.643236
TDF,-0.191357,1.216913
NTP,0.352123,0.315047
VLB,0.232584,0.112463
BMP,-0.276164,-0.100860
VCS,-0.044582,-0.201333
BCC,0.446905,-1.457062


## 6.4. <a id='toc6_4_'></a>[Thép](#toc0_)

In [ ]:
thep_pl = (
    nvl_pl[nvl_pl[GICS3] == "Thép"]
    .reset_index()
    .set_index(["Ticker", "fiscalDate"])
)
thep_pl


,,Tổng doanh thu hoạt động kinh doanh,Các khoản giảm trừ doanh thu,Doanh thu thuần,Giá vốn hàng bán,Lợi nhuận gộp,Doanh thu hoạt động tài chính,Chi phí tài chính,Trong đó: Chi phí lãi vay,Lợi nhuận hoặc lỗ trong công ty liên kết,Chi phí bán hàng,...,Chi phí thuế TNDN hiện hành,Chi phí thuế TNDN hoãn lại,Lợi nhuận sau thuế thu nhập doanh nghiệp,Lợi ích của cổ đông thiểu số,Lợi nhuận sau thuế của Công ty mẹ,Lãi cơ bản trên cổ phiếu,Lãi suy giảm trên cổ phiếu,GICS1_name,GICS2_name,GICS3_name
Ticker,fiscalDate,,,,,,,,,,,,,,,,,,,,,
AMC,2020-03-31,31.040374,0.000000,31.040374,17.103877,13.936498,0.029073,0.363479,0.362342,0.000000,10.120540,...,0.061746,0.000000,1.139044,0.000000,1.139044,3.330000e-07,3.330000e-07,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BCC,2020-03-31,1050.945519,0.000000,1050.945519,921.928505,129.017014,0.014527,20.872340,20.872340,0.000000,40.125413,...,6.507216,0.000000,18.575534,-1.615877,20.191411,1.840000e-07,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BDT,2020-03-31,109.083946,0.000000,109.083946,72.451059,36.632887,0.001538,2.858451,2.858451,0.000000,8.098063,...,4.021335,0.000000,16.092861,0.000000,16.092861,3.940000e-07,NaN,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BMP,2020-03-31,1020.368268,0.588220,1019.780048,774.099922,245.680126,17.505577,25.859785,0.009238,-0.116213,88.460624,...,25.284308,0.336890,102.368580,0.000000,102.368580,1.251000e-06,1.251000e-06,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
BTS,2020-03-31,688.661507,0.000000,688.661507,599.310724,89.350783,1.737259,22.433319,21.766638,0.000000,26.120832,...,3.938214,0.000000,11.447745,0.000000,11.447745,9.500000e-08,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VHL,2024-12-31,335.499383,0.000000,335.499383,296.285124,39.214260,0.985491,0.516961,0.468762,-3.625017,30.951928,...,1.153334,0.794371,-15.593147,0.000000,-15.593147,-6.280000e-07,-6.280000e-07,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
VIT,2024-12-31,707.806833,4.316474,703.490359,633.758583,69.731776,0.103792,23.226103,21.909737,0.000000,1.801414,...,7.537723,0.000000,35.292465,0.000000,35.292465,7.060000e-07,0.000000e+00,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng
VLB,2024-12-31,358.973686,0.000000,358.973686,265.978228,92.995458,12.609517,0.000000,0.000000,0.000000,3.745993,...,17.116436,-1.420922,62.635058,0.000000,62.635058,1.140000e-06,1.140000e-06,Nguyên vật liệu,Vật liệu xây dựng,Vật liệu xây dựng


In [ ]:
revni_thep = thep_pl[[REVENUE, NI]]
revni_thep

,,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,fiscalDate,,
AMC,2020-03-31,31.040374,1.139044
BCC,2020-03-31,1050.945519,18.575534
BDT,2020-03-31,109.083946,16.092861
BMP,2020-03-31,1019.780048,102.368580
BTS,2020-03-31,688.661507,11.447745
...,...,...,...
VHL,2024-12-31,335.499383,-15.593147
VIT,2024-12-31,703.490359,35.292465
VLB,2024-12-31,358.973686,62.635058


In [ ]:
top_DN_5 = revni_thep.xs(CURRENT_DATE, level="fiscalDate").sort_values(by=NI,ascending=False).head(10).index
top_DN_5

Index(['VGC', 'BMP', 'NTP', 'VCS', 'LIC', 'VLB', 'THG', 'BCC', 'VIT', 'TDF'], dtype='object', name='Ticker')

In [ ]:
thep_current_growth = current_yoy_growth(revni_thep)
thep_current_growth = thep_current_growth.sort_values(by=NI, ascending=False)
thep_current_growth_topDN = thep_current_growth[thep_current_growth.index.isin(top_DN_5)]
thep_current_growth_topDN

,Doanh thu thuần,Lợi nhuận sau thuế thu nhập doanh nghiệp
Ticker,,
VIT,0.253010,3.112319
LIC,0.515811,2.917338
THG,0.403528,1.643236
TDF,-0.191357,1.216913
NTP,0.352123,0.315047
VLB,0.232584,0.112463
BMP,-0.276164,-0.100860
VCS,-0.044582,-0.201333
BCC,0.446905,-1.457062
